# Interpretable Medical Anomaly Detection using Discriminatively Trained Part-Based Models

**What this notebook does.** It builds, from scratch, a *Deformable Part Model* (DPM, Felzenszwalb et al.) that
detects **cardiomegaly** (an enlarged heart) on chest X-rays from the **VinBigData Chest X-Ray Abnormalities
Detection** dataset (resized-PNG release).

**The approach in 4 sentences.** Every image is converted to a grid of HOG (Histogram of Oriented Gradients)
features. A *root filter* (a linear SVM template) slides over an image pyramid and scores every window for
"looks like an enlarged heart". Four *part filters* (heart borders / contours) then re-score the best windows and are
allowed to move a little, paying a learned quadratic "spring" penalty. Because the final score is a sum of named terms
(root + 4 parts − deformation + bias), every decision can be explained.

**Compute.** PyTorch is used **only as a tensor / optimisation library** (HOG, convolution scoring, SVM training).
There are **no neural networks** here. Set `DEVICE` to `'cpu'` to run without a GPU (slower).

**Notebook map:** 1 Setup · 2 Data & labels · 3 Aspect restoration & cache · 4 HOG · 5 Pyramid & root window ·
6 Training data · 7 Root filter · 8 Latent SVM (parts) · 9 Detection engine · 10 Metrics · 11 Interpretability ·
12 Baselines · 13 Compute profile · 14 Full report

## Section 1 — Setup & Imports

This cell installs any missing package, imports everything, defines **all** tunable constants in one place
(the rest of the notebook only references these names), creates the output folders, prints device information and
checks that the expected data files exist.

In [ ]:
# ---------------------------------------------------------------
# 1. Install missing packages (only if needed)
# ---------------------------------------------------------------
import importlib, subprocess, sys

_REQUIRED = {  # import name -> pip name
    'torch': 'torch', 'cv2': 'opencv-python', 'skimage': 'scikit-image', 'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib', 'numpy': 'numpy', 'pandas': 'pandas', 'joblib': 'joblib',
    'PIL': 'Pillow', 'tqdm': 'tqdm', 'psutil': 'psutil',
}
for _imp, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_imp)
    except ImportError:
        print(f'Installing {_pip} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pip])

# ---------------------------------------------------------------
# 2. Imports
# ---------------------------------------------------------------
import os, time, copy, json, math, warnings, platform
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
import joblib
import psutil
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.auto import tqdm
from skimage.feature import hog as skimage_hog          # used ONLY for visualisation / parity testing
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
                             matthews_corrcoef, roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve, confusion_matrix)

warnings.filterwarnings('ignore', category=UserWarning)
plt.rcParams['figure.dpi'] = 100

# ---------------------------------------------------------------
# 3. GLOBAL CONFIGURATION CONSTANTS
# ---------------------------------------------------------------
# ---- Paths ----
DATA_DIR        = Path('./data')
TRAIN_DIR       = DATA_DIR / 'train'
TEST_DIR        = DATA_DIR / 'test'
CSV_PATH        = DATA_DIR / 'train.csv'
TRAIN_META_PATH = DATA_DIR / 'train_meta.csv'
TEST_META_PATH  = DATA_DIR / 'test_meta.csv'
CACHE_DIR       = Path('./cache')
PKL_DIR         = Path('./pkl')
OUTPUT_DIR      = Path('./output')
PLOTS_DIR       = OUTPUT_DIR / 'plots'
REPORTS_DIR     = OUTPUT_DIR / 'reports'
TEST_IMAGE_PATH = None            # optional: set to a specific test PNG path for Section 14

# ---- Device ----
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'   # force 'cpu' to test the fallback
STORE_DTYPE     = torch.float16   # dtype for big stored feature matrices (compute is always float32)

# ---- Target & labels ----
TARGET_CLASS    = 'Cardiomegaly'
TARGET_ID       = 3
NO_FINDING_ID   = 14
MIN_RAD_VOTES   = 2               # radiologists needed for a positive label

# ---- Image ----
SRC_SIZE        = 1024            # PNG side length (squashed square)
CACHE_SIZE      = 512             # aspect-restored, letterboxed cache size
DET_DOWNSCALE   = 2               # root-filter level = CACHE_SIZE / 2 (256 px)
USE_CLAHE       = True
CLAHE_CLIP      = 2.0
CLAHE_TILE      = (8, 8)
PAD_VALUE       = 0               # letterbox fill
CACHE_WORKERS   = min(8, os.cpu_count() or 1)   # threads used while building the cache
BOUNDS_TOL_PX   = 5               # tolerance (original px) when checking boxes lie inside the image
BOUNDS_PASS_FRAC = 0.99           # fraction of boxes that must pass the bounds check

# ---- HOG ----
HOG_ORIENT      = 9
HOG_PPC         = (8, 8)          # pixels per cell
HOG_CPB         = (2, 2)          # cells per block
HOG_PARITY_MIN  = 0.90            # warn if correlation with skimage HOG falls below this

# ---- Root window (derived from data at runtime, bounded by these) ----
ROOT_SIZE_PERCENTILE = 25         # window sized to the small end of heart sizes
ROOT_CONTEXT_MARGIN  = 0.10       # extra lung-field context around the box
ROOT_MIN_CELLS       = 8
ROOT_MAX_CELLS       = 20
COVERAGE_MIN         = 0.95       # recommended minimum scale-range coverage of positive boxes

# ---- Pyramid ----
N_SCALES        = 5
SCALE_FACTOR    = 0.85            # 0.85^4 ~ 0.52 -> covers ~1.9x size range

# ---- Detection ----
STRIDE_CELLS    = 1               # dense: every HOG cell
ROOT_TOP_K      = 20              # root-score candidates per image sent to part rescoring
LOCAL_MAX_KERNEL = 3              # candidates must be local maxima of the score map in this neighbourhood
THRESHOLD       = 0.0             # decision threshold on final score
IOU_NMS         = 0.3
NO_DET_SCORE    = -10.0           # image score used when an image has no valid window
USE_POSITION_PRIOR   = True       # restrict search to where hearts occur in training
PRIOR_PERCENTILES    = (1, 99)    # of positive box centres

# ---- Model / training ----
N_PARTS         = 4
PART_RES_FACTOR = 2               # parts use HOG at 2x the root resolution
PART_SEARCH_CELLS = 2             # max part displacement (in part-level cells)
PART_MIN_CELLS  = 3               # smallest allowed part window (cells) -> at least 2x2 blocks
DEFORM_INIT_QUAD  = 0.01          # initial quadratic deformation weight
SVM_C           = 0.01
SVM_MAX_ITER    = 200
SVM_PROJ_ROUNDS = 4               # L-BFGS is restarted this many times; non-negativity is projected in between
SVM_CHUNK       = 8192            # rows per chunk when evaluating the SVM objective
LSVM_ITER       = 5
LSVM_BIAS_SCALE = 10.0            # scale of the constant '1' feature (keeps the bias lightly regularised)
LSVM_NEG_IMAGES = 4000            # negative images whose hard windows feed the latent SVM
LSVM_NEG_PER_IMAGE = 3
NEG_PER_IMAGE   = 5               # random negative windows per negative image
INIT_NEG_IMAGES = 4000            # initial negative images (None = all)
MINE_NEG_IMAGES = None            # negative images scanned per hard-negative round (None = all)
HARD_NEG_ROUNDS = 3
HARD_NEG_PER_IMAGE = 5
HARD_NEG_MARGIN = -1.0            # windows scoring above this are margin violators (hard)
POS_JITTER_CELLS = 1              # positive augmentation (shift only; NO horizontal flip)
NEG_IOU_MAX     = 0.3             # random negative windows in positive images must have IoU below this
NEG_TRIES       = 30              # rejection-sampling attempts per random negative window
SANITY_SUBSET   = 2000            # rows for the LinearSVC sanity check
VAL_SPLIT       = 0.2
RANDOM_SEED     = 42


# ---- Baseline 1 (image-level global HOG) ----
B1_SIZE         = 128             # whole image is resized to B1_SIZE x B1_SIZE before HOG
B1_C            = SVM_C

# ---- Reporting ----
HIGH_DEFORM_THRESH = 0.05
CONF_HIGH       = 0.5             # final score >= this -> HIGH confidence
BENCH_IMAGES    = 20
BENCH_CPU_IMAGES = 3              # subset used for the CPU-only timing comparison
ASCII_BAR_WIDTH = 30
MODEL_VERSION   = 'DPM-v2.0'
REPORT_ID       = datetime.now().strftime('%Y%m%d_%H%M%S')

# ---- Derived helpers ----
CELL_H, CELL_W  = HOG_PPC
HOG_BLOCK_DIM   = HOG_CPB[0] * HOG_CPB[1] * HOG_ORIENT        # 36 numbers per block
ROOT_LEVEL_SIZE = CACHE_SIZE // DET_DOWNSCALE                  # side of the root-level (pyramid level 0) image
PART_NAMES  = ['Right cardiac border (pt. right)', 'Left cardiac border (pt. left)',
               'Superior contour', 'Inferior contour']
PART_COLORS = ['cyan', 'lime', 'orange', 'magenta']
# (rel_x, rel_y, width_factor, height_factor) relative to the root BOX (cache space). Initial priors only.
PART_ANCHORS = [
    (0.0, 0.2, 0.3, 0.6),   # Part 0: image-left edge  = PATIENT'S RIGHT heart border
    (0.7, 0.2, 0.3, 0.6),   # Part 1: image-right edge = PATIENT'S LEFT heart border
    (0.1, 0.0, 0.8, 0.3),   # Part 2: superior contour (great vessels / aortic arch region)
    (0.1, 0.7, 0.8, 0.3),   # Part 3: inferior contour (diaphragmatic surface)
]
assert len(PART_ANCHORS) == N_PARTS == len(PART_NAMES) == len(PART_COLORS)

# ---- Create folders, seeds ----
for _d in (CACHE_DIR, PKL_DIR, OUTPUT_DIR, PLOTS_DIR, REPORTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ---- Device info ----
print('=' * 70)
print(f'Python {platform.python_version()} | torch {torch.__version__} | OpenCV {cv2.__version__}')
print(f'DEVICE = {DEVICE}')
if DEVICE == 'cuda':
    _p = torch.cuda.get_device_properties(0)
    print(f'GPU    = {_p.name} | VRAM = {_p.total_memory / 1024**3:.1f} GB')
else:
    print('Running on CPU (all steps work, but training and evaluation will be much slower).')
print(f'Host RAM = {psutil.virtual_memory().total / 1024**3:.1f} GB | CPU threads = {os.cpu_count()}')
print('=' * 70)

# ---- Path-existence table ----
_paths = {'TRAIN_DIR': TRAIN_DIR, 'TEST_DIR': TEST_DIR, 'CSV_PATH': CSV_PATH,
          'TRAIN_META_PATH': TRAIN_META_PATH, 'TEST_META_PATH': TEST_META_PATH}
print(pd.DataFrame({'path': [str(v) for v in _paths.values()],
                    'exists': [v.exists() for v in _paths.values()]}, index=list(_paths)).to_string())
for _name in ('TRAIN_DIR', 'CSV_PATH', 'TRAIN_META_PATH'):
    if not _paths[_name].exists():
        raise FileNotFoundError(f'{_name} = {_paths[_name]} is required but missing. '
                                'Download the dataset (see the project plan) into ./data/.')
if not TEST_META_PATH.exists():
    print('\nNOTE: test_meta.csv not found -> Section 14 will fall back to a validation image.')

## Section 2 — Data Loading, Labels & Box-Scaling Verification

**The CSV.** `train.csv` has one row per radiologist box: `image_id, class_name, class_id, rad_id, x_min, y_min, x_max, y_max`.
Three radiologists annotate every training image. Coordinates are in the **original DICOM pixel space**
(roughly 900–3400 px per side), *not* in the 1024×1024 PNG space. `train_meta.csv` gives each image's original
size: `dim0` = height, `dim1` = width.

**Cardiomegaly** is an enlarged cardiac silhouette; clinically the cardiothoracic ratio (heart width / thorax width)
is above about 0.5. It is a *global* measurement, which is hard for a local template, so expect modest numbers.

**Labels from several radiologists.** Radiologists disagree. We call an image
- **positive** if ≥ `MIN_RAD_VOTES` (2) distinct radiologists drew a Cardiomegaly box,
- **ambiguous** if exactly 1 did (excluded from training and validation, but tabulated below),
- **negative** if nobody did. This deliberately includes images with *other* findings (effusion, nodules, ...).

The vote threshold exists because a single-radiologist label is noisy; training a detector on it would teach it that noise.
The master list of images is `train_meta.csv` (not the unique ids in `train.csv`).

In [ ]:
# ---------------- Load & explore ----------------
train_df = pd.read_csv(CSV_PATH)
meta_df  = pd.read_csv(TRAIN_META_PATH)

# Runtime assumption checks (fail loudly)
_need = {'image_id', 'class_name', 'class_id', 'rad_id', 'x_min', 'y_min', 'x_max', 'y_max'}
assert _need <= set(train_df.columns), f'train.csv is missing columns: {_need - set(train_df.columns)}'
assert {'image_id', 'dim0', 'dim1'} <= set(meta_df.columns), 'train_meta.csv must have image_id, dim0, dim1'
assert meta_df['image_id'].is_unique, 'train_meta.csv has duplicate image_id rows'

png_ids = {p.stem for p in TRAIN_DIR.glob('*.png')}
if not png_ids:
    raise FileNotFoundError(f'No PNG files found in {TRAIN_DIR}')
missing_meta = png_ids - set(meta_df['image_id'])
assert not missing_meta, (f'{len(missing_meta)} training PNGs have no row in train_meta.csv '
                          f'(e.g. {sorted(missing_meta)[:3]}). Aspect restoration needs original sizes.')
missing_png = set(meta_df['image_id']) - png_ids
if missing_png:
    print(f'WARNING: {len(missing_png)} ids in train_meta.csv have no PNG on disk; they are dropped.')

# PNG format check on a few files: 1024x1024, mode L
for _p in sorted(TRAIN_DIR.glob('*.png'))[:5]:
    with Image.open(_p) as _im:
        if _im.size != (SRC_SIZE, SRC_SIZE):
            raise ValueError(f'{_p.name}: size {_im.size}, expected {(SRC_SIZE, SRC_SIZE)}')
        if _im.mode != 'L':
            print(f'NOTE: {_p.name} has mode {_im.mode}; the Preprocessor converts it to grayscale.')

print(f'train.csv rows            : {len(train_df):,}')
print(f'unique images in train.csv: {train_df.image_id.nunique():,}')
print(f'train_meta.csv rows       : {len(meta_df):,}   (PNGs on disk: {len(png_ids):,})')
print(f'classes                   : {train_df.class_name.nunique()}')

# Class distribution (images per class, excluding the placeholder rows of "No finding")
class_counts = (train_df.groupby('class_name')['image_id'].nunique().sort_values(ascending=False))
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['crimson' if c == TARGET_CLASS else 'steelblue' for c in class_counts.index]
ax.bar(range(len(class_counts)), class_counts.values, color=colors)
ax.set_xticks(range(len(class_counts)))
ax.set_xticklabels(class_counts.index, rotation=60, ha='right')
ax.set_ylabel('Number of images'); ax.set_xlabel('Class')
ax.set_title(f'Images per class (target = {TARGET_CLASS} in red)')
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'class_distribution.png', dpi=120); plt.show()

# ---------------- Votes per image ----------------
cardio_rows = train_df[(train_df.class_id == TARGET_ID)].copy()
assert (cardio_rows.class_name == TARGET_CLASS).all(), 'TARGET_ID does not match TARGET_CLASS in train.csv'
votes = cardio_rows.groupby('image_id')['rad_id'].nunique()

other_findings = set(train_df.loc[~train_df.class_id.isin([TARGET_ID, NO_FINDING_ID]), 'image_id'])
all_ids = meta_df[meta_df.image_id.isin(png_ids)].copy()
all_ids['n_rads'] = all_ids.image_id.map(votes).fillna(0).astype(int)
all_ids['label'] = np.where(all_ids.n_rads >= MIN_RAD_VOTES, 'positive',
                     np.where(all_ids.n_rads == 1, 'ambiguous', 'negative'))
all_ids['has_other'] = all_ids.image_id.isin(other_findings)

_neg = all_ids[all_ids.label == 'negative']
_amb = all_ids[all_ids.label == 'ambiguous']
label_table = pd.DataFrame({
    'images': [(all_ids.label == 'positive').sum(), len(_amb), len(_neg)],
    'with other findings': ['-', int(_amb.has_other.sum()), int(_neg.has_other.sum())],
    'note': [f'>= {MIN_RAD_VOTES} radiologists', 'exactly 1 radiologist (EXCLUDED)',
             f'0 radiologists ({int((~_neg.has_other).sum())} are "No finding")']},
    index=['positive', 'ambiguous', 'negative'])
print('\nLABEL TABLE'); print(label_table.to_string())
print('\nRadiologist votes for Cardiomegaly (images):'); print(all_ids.n_rads.value_counts().sort_index().to_string())

In [ ]:
def get_consensus_bbox(df, image_id):
    """Consensus Cardiomegaly box for one image, in ORIGINAL pixel space.

    Each radiologist's boxes are merged into their enclosing box; the per-radiologist boxes are then averaged.
    Returns (x_min, y_min, x_max, y_max) or None if the image has no Cardiomegaly rows.
    """
    rows = df[(df.image_id == image_id) & (df.class_id == TARGET_ID)]
    if rows.empty:
        return None
    per_rad = rows.groupby('rad_id').agg(x1=('x_min', 'min'), y1=('y_min', 'min'),
                                         x2=('x_max', 'max'), y2=('y_max', 'max'))
    return tuple(float(v) for v in per_rad[['x1', 'y1', 'x2', 'y2']].mean().values)

# Fast path: group the target rows once instead of filtering the full table per image
_cardio_by_img = {k: g for k, g in cardio_rows.groupby('image_id')}
def _consensus_fast(image_id):
    g = _cardio_by_img.get(image_id)
    return None if g is None else get_consensus_bbox(g, image_id)

keep = all_ids[all_ids.label.isin(['positive', 'negative'])].copy()
boxes = [(_consensus_fast(i) if lab == 'positive' else None) for i, lab in zip(keep.image_id, keep.label)]
records = pd.DataFrame({
    'image_id': keep.image_id.values,
    'png_path': [str(TRAIN_DIR / f'{i}.png') for i in keep.image_id],
    'has_target': (keep.label == 'positive').astype(int).values,
    'n_rads': keep.n_rads.values,
    'H': keep.dim0.astype(int).values, 'W': keep.dim1.astype(int).values,
    'has_other': keep.has_other.values,
    'x1': [b[0] if b else np.nan for b in boxes], 'y1': [b[1] if b else np.nan for b in boxes],
    'x2': [b[2] if b else np.nan for b in boxes], 'y2': [b[3] if b else np.nan for b in boxes],
})
ambiguous_df = all_ids[all_ids.label == 'ambiguous'].copy()
print(f'records: {len(records):,} images ({records.has_target.sum():,} positive, '
      f'{(1 - records.has_target).sum():,} negative); {len(ambiguous_df):,} ambiguous excluded')

# Box size as a fraction of the ORIGINAL image (drives root-window sizing later)
_pos = records[records.has_target == 1]
_fw, _fh = (_pos.x2 - _pos.x1) / _pos.W, (_pos.y2 - _pos.y1) / _pos.H
print('\nCardiomegaly box size as a fraction of the original image')
print(pd.DataFrame({'width_frac': _fw.describe(percentiles=[.05, .25, .5, .75, .95]),
                    'height_frac': _fh.describe(percentiles=[.05, .25, .5, .75, .95])}).round(3).to_string())

In [ ]:
# ---------------- Verify scaling (HARD GATE) ----------------
def bounds_pass_fraction(rec, swap=False):
    """Fraction of positive boxes inside (W, H) in original space (optionally with dim0/dim1 swapped)."""
    W_ = rec.H if swap else rec.W
    H_ = rec.W if swap else rec.H
    ok = ((rec.x1 >= -BOUNDS_TOL_PX) & (rec.y1 >= -BOUNDS_TOL_PX) &
          (rec.x2 <= W_ + BOUNDS_TOL_PX) & (rec.y2 <= H_ + BOUNDS_TOL_PX))
    return float(ok.mean())

frac_ok = bounds_pass_fraction(_pos)
print(f'Boxes inside the image (x_max <= dim1, y_max <= dim0, tol {BOUNDS_TOL_PX}px): {100 * frac_ok:.2f}%')
if frac_ok < BOUNDS_PASS_FRAC:
    frac_swapped = bounds_pass_fraction(_pos, swap=True)
    raise RuntimeError(
        f'Box bounds check FAILED ({100 * frac_ok:.1f}% < {100 * BOUNDS_PASS_FRAC:.0f}%). '
        f'With dim0/dim1 swapped the pass rate would be {100 * frac_swapped:.1f}%. '
        'If that is much higher, dim0/dim1 are swapped in your train_meta.csv; fix the file (or swap H and W above).')

# Visual check: map ORIGINAL -> PNG space (x * 1024/W, y * 1024/H); the box must sit on the heart.
_sel = _pos.sample(8, random_state=RANDOM_SEED)
fig, axes = plt.subplots(2, 4, figsize=(16, 8.5))
for ax, (_, r) in zip(axes.ravel(), _sel.iterrows()):
    im = np.array(Image.open(r.png_path).convert('L'))
    ax.imshow(im, cmap='gray')
    bx1, bx2 = r.x1 * SRC_SIZE / r.W, r.x2 * SRC_SIZE / r.W
    by1, by2 = r.y1 * SRC_SIZE / r.H, r.y2 * SRC_SIZE / r.H
    ax.add_patch(mpatches.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, fill=False, ec='lime', lw=2))
    ax.set_title(f'{r.image_id[:8]}  ({r.W}x{r.H})', fontsize=9); ax.axis('off')
fig.suptitle('Scaling check: consensus Cardiomegaly boxes mapped original -> PNG space (must sit on the heart)')
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'box_scaling_check.png', dpi=120); plt.show()

In [ ]:
# ---------------- Train / Val split (stratified; ambiguous already excluded) ----------------
train_rec, val_rec = train_test_split(records, test_size=VAL_SPLIT, stratify=records.has_target,
                                      random_state=RANDOM_SEED)
train_rec = train_rec.reset_index(drop=True)
val_rec   = val_rec.reset_index(drop=True)      # keeps its NATURAL class ratio
print(pd.DataFrame({
    'images':   [len(train_rec), len(val_rec)],
    'positive': [int(train_rec.has_target.sum()), int(val_rec.has_target.sum())],
    'negative': [int((1 - train_rec.has_target).sum()), int((1 - val_rec.has_target).sum())],
    'positive %': [100 * train_rec.has_target.mean(), 100 * val_rec.has_target.mean()]},
    index=['train', 'val']).round(2).to_string())

# ---------------- Sample visualisation: 4 positives + 4 negatives ----------------
def show_png_samples(rec_pos, rec_neg, save_name):
    """Plot 2x4 grid (top: positives with box, bottom: negatives) in PNG space."""
    fig, axes = plt.subplots(2, 4, figsize=(16, 8.5))
    for row, (df_, title) in enumerate(((rec_pos, 'Cardiomegaly'), (rec_neg, 'No Cardiomegaly'))):
        for ax, (_, r) in zip(axes[row], df_.iterrows()):
            ax.imshow(np.array(Image.open(r.png_path).convert('L')), cmap='gray')
            if r.has_target:
                ax.add_patch(mpatches.Rectangle((r.x1 * SRC_SIZE / r.W, r.y1 * SRC_SIZE / r.H),
                             (r.x2 - r.x1) * SRC_SIZE / r.W, (r.y2 - r.y1) * SRC_SIZE / r.H,
                             fill=False, ec='lime', lw=2))
            ax.set_title(f'{title}' + (' (other findings)' if (not r.has_target and r.has_other) else ''), fontsize=9)
            ax.axis('off')
    fig.suptitle('Sample training images (PNG space, aspect still squashed)')
    plt.tight_layout(); plt.savefig(PLOTS_DIR / save_name, dpi=120); plt.show()

show_png_samples(train_rec[train_rec.has_target == 1].sample(4, random_state=1),
                 train_rec[train_rec.has_target == 0].sample(4, random_state=1), 'sample_images.png')

## Section 3 — PNG Loading, Aspect Restoration & Caching

**Why restore the aspect ratio?** The dataset authors resized every original (e.g. 2800×2300) to a 1024×1024 square
by *stretching* it. About 87 % of originals are not square (width/height has mean 0.88), so each heart is distorted by a
*different* amount. A HOG template learns oriented edges, so a heart that was squeezed horizontally by 12 % in one image
and 4 % in another looks like two different shapes. We undo the stretch using the original size from `train_meta.csv`.

**Letterboxing.** After resizing to `(new_h, new_w)` with one uniform scale `s = CACHE_SIZE / max(H, W)`, the
image is pasted into the middle of a `CACHE_SIZE × CACHE_SIZE` black canvas. All modelling happens in this *cache space*.
Boxes are converted original → cache **exactly once** with `x_cache = x_orig · s + pad_x`.

The dataset authors already applied DICOM windowing and MONOCHROME1 inversion, so this notebook does **not** repeat them.
CLAHE (local contrast equalisation) is applied only to the image content, not to the black padding.

In [ ]:
class Preprocessor:
    """PNG -> aspect-restored, letterboxed, contrast-enhanced uint8 image of size CACHE_SIZE x CACHE_SIZE."""

    def __init__(self, cache_size=CACHE_SIZE, use_clahe=USE_CLAHE):
        """Store the output size and build the CLAHE operator."""
        self.cache_size = cache_size
        self.use_clahe = use_clahe
        self.clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)

    @staticmethod
    def geometry(H, W, cache_size=CACHE_SIZE):
        """Return (s, new_w, new_h, pad_x, pad_y) of the letterboxed placement for an original H x W image."""
        if H is None or W is None or not np.isfinite(H) or not np.isfinite(W) or H <= 0 or W <= 0:
            raise ValueError(f'Invalid original size (H={H}, W={W}); missing row in the meta CSV?')
        s = cache_size / max(H, W)
        new_w, new_h = max(1, int(round(W * s))), max(1, int(round(H * s)))
        return s, new_w, new_h, (cache_size - new_w) // 2, (cache_size - new_h) // 2

    def load_png(self, path):
        """Load a PNG as uint8 grayscale (1024x1024 expected). Handles 3/4-channel and 16-bit inputs."""
        img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        if img is None or img.size == 0:
            raise IOError(f'Could not read image (missing or empty): {path}')
        if img.ndim == 3:
            if img.shape[2] == 4:
                img = cv2.cvtColor(img, cv2.COLOR_BGRA2GRAY)
            elif img.shape[2] == 3:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            else:
                raise ValueError(f'Unexpected channel count {img.shape[2]} in {path}')
        if img.dtype != np.uint8:      # e.g. 16-bit PNG
            img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        return img

    def restore_aspect(self, img, H, W):
        """Resize the squashed PNG straight to (new_h, new_w) (undoes the stretch AND downsamples), then letterbox.

        Returns (cache_img, s, pad_x, pad_y).
        """
        s, new_w, new_h, pad_x, pad_y = self.geometry(H, W, self.cache_size)
        interp = cv2.INTER_AREA if (new_w <= img.shape[1] and new_h <= img.shape[0]) else cv2.INTER_LINEAR
        resized = cv2.resize(img, (new_w, new_h), interpolation=interp)
        canvas = np.full((self.cache_size, self.cache_size), PAD_VALUE, dtype=np.uint8)
        canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized
        return canvas, s, pad_x, pad_y

    def apply_clahe(self, img, region=None):
        """CLAHE on `img`. If region=(y0, y1, x0, x1) is given only that part is equalised (letterbox stays black)."""
        if not self.use_clahe:
            return img
        out = img.copy()
        if region is None:
            return self.clahe.apply(out)
        y0, y1, x0, x1 = region
        out[y0:y1, x0:x1] = self.clahe.apply(np.ascontiguousarray(out[y0:y1, x0:x1]))
        return out

    def preprocess(self, path, H, W):
        """Full pipeline. Returns (cache_img uint8, meta dict with H, W, s, pad_x, pad_y)."""
        img = self.load_png(path)
        cache_img, s, pad_x, pad_y = self.restore_aspect(img, H, W)
        _, new_w, new_h, _, _ = self.geometry(H, W, self.cache_size)
        cache_img = self.apply_clahe(cache_img, (pad_y, pad_y + new_h, pad_x, pad_x + new_w))
        return cache_img, {'H': int(H), 'W': int(W), 's': s, 'pad_x': pad_x, 'pad_y': pad_y}

    @staticmethod
    def map_box(bbox_orig, s, pad_x, pad_y):
        """Original-space box (x1, y1, x2, y2) -> cache-space box. Call this exactly once per box."""
        if bbox_orig is None:
            return None
        x1, y1, x2, y2 = bbox_orig
        return (x1 * s + pad_x, y1 * s + pad_y, x2 * s + pad_x, y2 * s + pad_y)

    @staticmethod
    def unmap_box(bbox_cache, s, pad_x, pad_y, H=None, W=None):
        """Cache-space box -> original-space box (used in the report). Clipped to the image if H, W are given."""
        x1, y1, x2, y2 = bbox_cache
        out = [(x1 - pad_x) / s, (y1 - pad_y) / s, (x2 - pad_x) / s, (y2 - pad_y) / s]
        if H is not None and W is not None:
            out = [min(max(out[0], 0), W), min(max(out[1], 0), H), min(max(out[2], 0), W), min(max(out[3], 0), H)]
        return tuple(out)

PRE = Preprocessor()
print('Preprocessor ready. Example geometry for a 2800 x 2300 (H x W) original:',
      tuple(round(v, 3) for v in PRE.geometry(2800, 2300)))

In [ ]:
# ---------------- Build the cache (resumable) ----------------
CACHE_INDEX_PATH  = CACHE_DIR / 'cache_index.csv'
CACHE_CONFIG_PATH = CACHE_DIR / 'cache_config.json'
_cfg = {'CACHE_SIZE': CACHE_SIZE, 'USE_CLAHE': USE_CLAHE, 'CLAHE_CLIP': CLAHE_CLIP,
        'CLAHE_TILE': list(CLAHE_TILE), 'PAD_VALUE': PAD_VALUE}
if CACHE_CONFIG_PATH.exists() and json.loads(CACHE_CONFIG_PATH.read_text()) != _cfg:
    print('Cache settings changed since the last run -> clearing the stale cache.')
    for _f in CACHE_DIR.rglob('*.npy'):
        _f.unlink()
CACHE_CONFIG_PATH.write_text(json.dumps(_cfg))
for _s in ('train', 'test'):
    (CACHE_DIR / _s).mkdir(exist_ok=True)

def cache_path(image_id, split):
    """Path of the cached uint8 array for an image."""
    return CACHE_DIR / split / f'{image_id}.npy'

def load_cached(image_id, split='train'):
    """Load a cached CACHE_SIZE x CACHE_SIZE uint8 image."""
    return np.load(cache_path(image_id, split))

def _cache_one(job):
    """Cache a single image (skips if it already exists). Returns 'done', 'skipped' or an error string."""
    image_id, png_path, H, W, split = job
    out = cache_path(image_id, split)
    if out.exists():
        return 'skipped'
    try:
        img, _ = PRE.preprocess(png_path, H, W)
        tmp = out.with_suffix('.tmp.npy')
        np.save(tmp, img)
        os.replace(tmp, out)           # atomic -> safe to interrupt and resume
        return 'done'
    except Exception as e:             # keep going; report at the end
        return f'ERROR {image_id}: {e}'

# Jobs: train/val records (ambiguous images are never used) + test images that have meta rows
jobs = [(r.image_id, r.png_path, r.H, r.W, 'train') for r in records.itertuples()]
test_meta_df = None
if TEST_META_PATH.exists():
    test_meta_df = pd.read_csv(TEST_META_PATH)
    assert {'image_id', 'dim0', 'dim1'} <= set(test_meta_df.columns), 'test_meta.csv needs image_id, dim0, dim1'
    _test_png = {p.stem for p in TEST_DIR.glob('*.png')}
    jobs += [(r.image_id, str(TEST_DIR / f'{r.image_id}.png'), r.dim0, r.dim1, 'test')
             for r in test_meta_df.itertuples() if r.image_id in _test_png]
else:
    print('test_meta.csv missing: test images are not cached (Section 14 will use a validation image).')

with ThreadPoolExecutor(CACHE_WORKERS) as ex:
    status = list(tqdm(ex.map(_cache_one, jobs), total=len(jobs), desc='caching'))
errors = [s for s in status if s.startswith('ERROR')]
print(f'cache: {status.count("done"):,} written, {status.count("skipped"):,} already cached, {len(errors)} errors')
for e in errors[:5]:
    print('  ', e)
if errors:
    raise RuntimeError(f'{len(errors)} images failed to cache (first: {errors[0]})')

# Index with the geometry of every cached image (needed to map boxes back to original space)
_rows = []
for (image_id, _, H, W, split) in jobs:
    s, nw, nh, px, py = Preprocessor.geometry(H, W)
    _rows.append({'image_id': image_id, 'split': split, 'H': int(H), 'W': int(W), 's': s, 'pad_x': px, 'pad_y': py})
cache_index = pd.DataFrame(_rows)
cache_index.to_csv(CACHE_INDEX_PATH, index=False)

# Convert boxes original -> cache space EXACTLY ONCE and store them
def add_cache_boxes(df):
    """Add s, pad_x, pad_y and cache-space box columns (cx1, cy1, cx2, cy2). Refuses to run twice."""
    assert 'cx1' not in df.columns, 'boxes were already converted to cache space'
    df = df.copy()
    geo = np.array([Preprocessor.geometry(h, w) for h, w in zip(df.H, df.W)])
    df['s'], df['pad_x'], df['pad_y'] = geo[:, 0], geo[:, 3].astype(int), geo[:, 4].astype(int)
    df['cx1'] = df.x1 * df.s + df.pad_x; df['cx2'] = df.x2 * df.s + df.pad_x
    df['cy1'] = df.y1 * df.s + df.pad_y; df['cy2'] = df.y2 * df.s + df.pad_y
    return df

train_rec = add_cache_boxes(train_rec)
val_rec   = add_cache_boxes(val_rec)
_p = pd.concat([train_rec, val_rec]); _p = _p[_p.has_target == 1]
assert ((_p.cx1 >= -BOUNDS_TOL_PX) & (_p.cy1 >= -BOUNDS_TOL_PX) &
        (_p.cx2 <= CACHE_SIZE + BOUNDS_TOL_PX) & (_p.cy2 <= CACHE_SIZE + BOUNDS_TOL_PX)).mean() >= BOUNDS_PASS_FRAC, \
    'cache-space boxes fall outside the canvas: check the coordinate conversion'
print(f'train_rec: {len(train_rec):,} | val_rec: {len(val_rec):,} | cache-space boxes stored.')

In [ ]:
# ---------------- Visualise: squashed PNG | aspect restored | after CLAHE ----------------
_show = train_rec[train_rec.has_target == 1].sample(3, random_state=3)
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for row, (_, r) in zip(axes, _show.iterrows()):
    png = PRE.load_png(r.png_path)
    restored, s, px, py = PRE.restore_aspect(png, r.H, r.W)
    _, nw, nh, _, _ = Preprocessor.geometry(r.H, r.W)
    clahe_img = PRE.apply_clahe(restored, (py, py + nh, px, px + nw))
    box = (r.cx1, r.cy1, r.cx2, r.cy2)
    row[0].imshow(png, cmap='gray'); row[0].set_title(f'PNG (squashed) {r.W}x{r.H} -> 1024x1024', fontsize=9)
    row[1].imshow(restored, cmap='gray'); row[1].set_title(f'aspect restored (s={s:.3f}, pad=({px},{py}))', fontsize=9)
    row[2].imshow(clahe_img, cmap='gray'); row[2].set_title('after CLAHE', fontsize=9)
    for ax in row[1:]:
        ax.add_patch(mpatches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], fill=False, ec='lime', lw=2))
    for ax in row:
        ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')
fig.suptitle('Preprocessing: stretch removal, letterboxing and contrast enhancement (boxes in cache space)')
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'preprocessing_comparison.png', dpi=110); plt.show()

## Section 4 — HOG Feature Extraction (GPU)

**HOG in one paragraph.** The image is divided into small **cells** (8×8 pixels here). In every cell we histogram the
gradient directions (unsigned, **9 orientations** covering 0–180°), weighting each pixel by its gradient magnitude and
splitting each vote between the two nearest orientation bins (bilinear voting). Neighbouring cells are grouped
into overlapping **blocks** (2×2 cells); each block's 36 numbers are L2-Hys normalised (normalise → clip at 0.2 →
re-normalise) so the descriptor tolerates brightness and contrast changes.
8×8 cells are a compromise: big enough to hold a stable orientation histogram, small enough to see the curvature of a
heart border at 256–512 px.

**Dimensionality.** A detection window of `rh × rw` cells contains `(rh−1)(rw−1)` blocks, each with 2×2×9 = 36
numbers, i.e. `(rh−1)(rw−1) × 36` features.

**Computed once, then sliced.** `block_map()` returns a *dense* grid of block descriptors for a whole image (or pyramid level).
A detection window is simply a contiguous sub-grid of that map, so sliding a filter over the image becomes a single
convolution and we never re-extract HOG per patch.

In [ ]:
class HOGExtractor:
    """PyTorch HOG: dense block maps computed once per image / pyramid level."""

    def __init__(self, orientations=HOG_ORIENT, pixels_per_cell=HOG_PPC,
                 cells_per_block=HOG_CPB, device=DEVICE):
        """Store HOG parameters and the compute device."""
        self.n_orient = orientations
        self.ppc = tuple(pixels_per_cell)
        self.cpb = tuple(cells_per_block)
        self.device = device
        self.block_dim = self.cpb[0] * self.cpb[1] * self.n_orient

    # ---- helpers ---------------------------------------------------------------
    def blocks_in(self, cells_y, cells_x):
        """Number of blocks (rows, cols) for a grid of cells_y x cells_x cells."""
        return cells_y - self.cpb[0] + 1, cells_x - self.cpb[1] + 1

    def feature_dim(self, ch, cw):
        """Feature dimension of a window of ch x cw cells: (ch-1)(cw-1) * 2*2*9."""
        by, bx = self.blocks_in(ch, cw)
        return max(by, 0) * max(bx, 0) * self.block_dim

    def _to_tensor(self, imgs):
        """uint8/float numpy (H,W) or (B,H,W) -> float32 tensor (B,1,H,W) in [0,1] on self.device."""
        arr = np.asarray(imgs)
        scale = 255.0 if arr.dtype == np.uint8 else 1.0          # uint8 -> [0,1]; floats assumed already in [0,1]
        x = torch.as_tensor(arr).to(self.device)
        if x.ndim == 2:
            x = x.unsqueeze(0)
        return x.unsqueeze(1).float() / scale

    def _cell_histograms(self, x):
        """(B,1,H,W) -> (B, cells_y, cells_x, n_orient) magnitude-weighted orientation histograms."""
        B, _, H, W = x.shape
        gx = torch.zeros_like(x); gy = torch.zeros_like(x)
        gx[..., :, 1:-1] = x[..., :, 2:] - x[..., :, :-2]        # centred difference [-1, 0, 1]
        gy[..., 1:-1, :] = x[..., 2:, :] - x[..., :-2, :]        # (borders stay 0, as in skimage)
        mag = torch.sqrt(gx * gx + gy * gy)
        ang = torch.remainder(torch.atan2(gy, gx), math.pi)      # unsigned orientation in [0, pi)
        pos = ang / (math.pi / self.n_orient) - 0.5              # bin centres at 0.5, 1.5, ...
        b0 = torch.floor(pos)
        frac = pos - b0
        i0 = torch.remainder(b0.long(), self.n_orient)
        i1 = torch.remainder(b0.long() + 1, self.n_orient)
        hist = torch.zeros(B, self.n_orient, H, W, device=x.device)
        hist.scatter_add_(1, i0, (1 - frac) * mag)               # bilinear voting between 2 bins
        hist.scatter_add_(1, i1, frac * mag)
        cells = F.avg_pool2d(hist, kernel_size=self.ppc, stride=self.ppc) * (self.ppc[0] * self.ppc[1])
        return cells.permute(0, 2, 3, 1).contiguous()

    # ---- public API ------------------------------------------------------------
    def block_maps(self, imgs):
        """Batched block maps: (B,H,W) -> (B, n_block_y, n_block_x, block_dim)."""
        cells = self._cell_histograms(self._to_tensor(imgs))
        by, bx = self.blocks_in(cells.shape[1], cells.shape[2])
        if by < 1 or bx < 1:                                      # image smaller than one block
            return torch.zeros(cells.shape[0], 0, 0, self.block_dim, device=self.device)
        blocks = torch.cat([cells[:, i:i + by, j:j + bx, :]
                            for i in range(self.cpb[0]) for j in range(self.cpb[1])], dim=-1)
        eps = 1e-5
        blocks = blocks / torch.sqrt((blocks ** 2).sum(-1, keepdim=True) + eps ** 2)      # L2
        blocks = blocks.clamp(max=0.2)                                                    # Hys clip
        return blocks / torch.sqrt((blocks ** 2).sum(-1, keepdim=True) + eps ** 2)        # re-normalise

    def block_map(self, img):
        """Single image (H,W) -> (n_block_y, n_block_x, block_dim). Computed ONCE per image / level."""
        return self.block_maps(img)[0]

    def window(self, block_map, cy, cx, ch, cw):
        """Flat feature vector of the window whose top-left CELL is (cy, cx) and size is ch x cw cells.

        Returns None if the window does not fit inside the block map (border).
        """
        bh, bw = self.blocks_in(ch, cw)
        if cy < 0 or cx < 0 or bh < 1 or bw < 1 or cy + bh > block_map.shape[0] or cx + bw > block_map.shape[1]:
            return None
        return block_map[cy:cy + bh, cx:cx + bw].reshape(-1)

    def visualise(self, img):
        """HOG image for DISPLAY ONLY (uses skimage). img: uint8 (H,W). Returns a float image."""
        _, hog_img = skimage_hog(np.asarray(img), orientations=self.n_orient, pixels_per_cell=self.ppc,
                                 cells_per_block=self.cpb, block_norm='L2-Hys', visualize=True)
        return hog_img

HOG = HOGExtractor()
print(f'HOGExtractor on {HOG.device}: block dim = {HOG.block_dim}')
print(f'Example: a 16x14-cell window has {HOG.feature_dim(16, 14):,} features = (16-1)*(14-1)*{HOG.block_dim}')

In [ ]:
# ---------------- Parity test against skimage.feature.hog on 5 images ----------------
_ids = train_rec.image_id.values[:5]
corrs = []
for _id in _ids:
    im = load_cached(_id, 'train')
    mine = HOG.block_map(im).cpu().numpy()
    ref = skimage_hog(im, orientations=HOG_ORIENT, pixels_per_cell=HOG_PPC, cells_per_block=HOG_CPB,
                      block_norm='L2-Hys', feature_vector=False)
    ref = ref.reshape(ref.shape[0], ref.shape[1], -1)
    assert mine.shape == ref.shape, f'shape mismatch {mine.shape} vs {ref.shape}'
    corrs.append(np.corrcoef(mine.ravel(), ref.ravel())[0, 1])
print(pd.DataFrame({'image': [i[:10] for i in _ids], 'correlation vs skimage': np.round(corrs, 4)}).to_string(index=False))
print(f'mean correlation = {np.mean(corrs):.4f} (small differences expected: bilinear orientation voting)')
if np.mean(corrs) < HOG_PARITY_MIN:
    print(f'WARNING: parity below {HOG_PARITY_MIN}; check the HOG implementation.')

# ---------------- Visualise HOG: 2 positives + 2 negatives ----------------
_sel = pd.concat([train_rec[train_rec.has_target == 1].head(2), train_rec[train_rec.has_target == 0].head(2)])
fig, axes = plt.subplots(2, 4, figsize=(16, 8.5))
for k, (_, r) in enumerate(_sel.iterrows()):
    im = load_cached(r.image_id, 'train')
    a, b = axes[k // 2][(k % 2) * 2], axes[k // 2][(k % 2) * 2 + 1]
    a.imshow(im, cmap='gray'); a.set_title(('Cardiomegaly' if r.has_target else 'No Cardiomegaly') + ' (cached image)', fontsize=9)
    b.imshow(HOG.visualise(im), cmap='hot'); b.set_title('HOG heat map', fontsize=9)
    for ax in (a, b):
        ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')
fig.suptitle('HOG features: bright strokes = strong edges at that orientation')
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'hog_visualization.png', dpi=110); plt.show()
_bm = HOG.block_map(load_cached(train_rec.image_id[0], 'train'))
print(f'Whole-image block map for a {CACHE_SIZE}px image: {tuple(_bm.shape)} -> {_bm.shape[0] * _bm.shape[1] * _bm.shape[2]:,} numbers')

## Section 5 — Image Pyramid & Root Window Sizing

**Multiscale detection.** A fixed-size window can only recognise a heart of one size. We therefore build an **image
pyramid**: level 0 is the root-level image (`CACHE_SIZE / DET_DOWNSCALE` = 256 px) and every further level is
shrunk by `SCALE_FACTOR`. A big heart that does not fit the window at level 0 shrinks to fit at a coarser level.

**Root window size.** The pyramid only *shrinks* the image, so the window must be sized to the **small end** of the heart
sizes (the `ROOT_SIZE_PERCENTILE`-th percentile of box area, at the median aspect ratio, plus `ROOT_CONTEXT_MARGIN` of
lung field on each side), clamped to `[ROOT_MIN_CELLS, ROOT_MAX_CELLS]`. Larger hearts are then matched at coarser levels. Everything is
derived from **training positives only**.

**Position prior.** Hearts always lie in the central chest. We store the 1st–99th percentile region of positive box
centres and (optionally) only search windows centred inside it. That removes a lot of false positives for free.

In [ ]:
# ---------------- Derive the root window from training positives ----------------
_pos_tr = train_rec[train_rec.has_target == 1]
_bw = (_pos_tr.cx2 - _pos_tr.cx1) / DET_DOWNSCALE / CELL_W      # box width  in root-level cells
_bh = (_pos_tr.cy2 - _pos_tr.cy1) / DET_DOWNSCALE / CELL_H      # box height in root-level cells
_area = _bw * _bh
_aspect = float(np.median(_bw / _bh))
_a_small = float(np.percentile(_area, ROOT_SIZE_PERCENTILE))
_h0 = math.sqrt(_a_small / _aspect); _w0 = _h0 * _aspect
_expand = 1 + 2 * ROOT_CONTEXT_MARGIN
ROOT_CH = int(np.clip(round(_h0 * _expand), ROOT_MIN_CELLS, ROOT_MAX_CELLS))    # window height in cells
ROOT_CW = int(np.clip(round(_w0 * _expand), ROOT_MIN_CELLS, ROOT_MAX_CELLS))    # window width  in cells
ROOT_DIM = HOG.feature_dim(ROOT_CH, ROOT_CW)
ROOT_BH, ROOT_BW = HOG.blocks_in(ROOT_CH, ROOT_CW)                               # window size in blocks

print(f'Positive boxes (train): median aspect w/h = {_aspect:.2f}; '
      f'{ROOT_SIZE_PERCENTILE}th-percentile area = {_a_small:.0f} cells^2')
print(f'ROOT WINDOW : {ROOT_CH} x {ROOT_CW} cells (h x w) = {ROOT_CH * CELL_H} x {ROOT_CW * CELL_W} px at the root level '
      f'(= {ROOT_CH * CELL_H * DET_DOWNSCALE} x {ROOT_CW * CELL_W * DET_DOWNSCALE} px in the {CACHE_SIZE}px cache)')
print(f'FEATURE DIM : ({ROOT_CH}-1)*({ROOT_CW}-1)*{HOG.block_dim} = {ROOT_DIM:,}')

# Coverage check: which positive boxes (incl. margin) can be matched by some pyramid level?
_ratio = np.sqrt(_area * _expand ** 2 / (ROOT_CH * ROOT_CW))       # linear size of the box relative to the window
_hi = 1 / SCALE_FACTOR ** (N_SCALES - 1)
_tol = SCALE_FACTOR ** 0.5                                         # half a pyramid step of tolerance on each side
COVERAGE = float(((_ratio >= _tol) & (_ratio <= _hi / _tol)).mean())
print(f'COVERAGE    : {100 * COVERAGE:.1f}% of positive boxes have size within [1.0, {_hi:.2f}] x window '
      f'(half-step tolerance); {100 * (_ratio < _tol).mean():.1f}% are smaller, {100 * (_ratio > _hi / _tol).mean():.1f}% larger')
if COVERAGE < COVERAGE_MIN:
    print(f'  -> below {100 * COVERAGE_MIN:.0f}%. Recommendation: raise N_SCALES (covers larger hearts) and/or lower '
          'ROOT_SIZE_PERCENTILE (smaller window covers smaller hearts), then re-run from Section 1.')

# ---------------- Position prior (box centres of training positives, cache space) ----------------
_cxs = ((_pos_tr.cx1 + _pos_tr.cx2) / 2).values; _cys = ((_pos_tr.cy1 + _pos_tr.cy2) / 2).values
PRIOR_RECT = (float(np.percentile(_cxs, PRIOR_PERCENTILES[0])), float(np.percentile(_cxs, PRIOR_PERCENTILES[1])),
              float(np.percentile(_cys, PRIOR_PERCENTILES[0])), float(np.percentile(_cys, PRIOR_PERCENTILES[1])))
print(f'POSITION PRIOR ({"ON" if USE_POSITION_PRIOR else "OFF"}): window centres must lie in '
      f'x [{PRIOR_RECT[0]:.0f}, {PRIOR_RECT[1]:.0f}], y [{PRIOR_RECT[2]:.0f}, {PRIOR_RECT[3]:.0f}] (cache px)')

In [ ]:
def resize_to(img, h, w):
    """Resize a 2-D image to (h, w); INTER_AREA when shrinking, INTER_LINEAR when enlarging."""
    if img.shape[0] == h and img.shape[1] == w:
        return img
    shrinking = (h <= img.shape[0]) and (w <= img.shape[1])
    return cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA if shrinking else cv2.INTER_LINEAR)

def to_root_level(cache_img):
    """Cache image -> root-level image (pyramid level 0)."""
    return resize_to(cache_img, ROOT_LEVEL_SIZE, ROOT_LEVEL_SIZE)

class ImagePyramid:
    """Gaussian-style image pyramid that only SHRINKS the image."""

    def __init__(self, n_scales=N_SCALES, scale_factor=SCALE_FACTOR, min_cells=(ROOT_CH, ROOT_CW)):
        """Store pyramid parameters; levels smaller than `min_cells` (cells) are not built."""
        self.n_scales, self.scale_factor, self.min_cells = n_scales, scale_factor, min_cells

    def build(self, img):
        """Return list of (scale, level_img). Stops when a level would be smaller than the root window."""
        H, W = img.shape[:2]
        levels = []
        for k in range(self.n_scales):
            s = self.scale_factor ** k
            h, w = int(round(H * s)), int(round(W * s))
            if k > 0 and (h // CELL_H < self.min_cells[0] or w // CELL_W < self.min_cells[1]):
                break
            levels.append((s, img if k == 0 else resize_to(img, h, w)))
        return levels

PYRAMID = ImagePyramid()

# ---------------- Visualise the pyramid of one positive image ----------------
_r = train_rec[train_rec.has_target == 1].iloc[0]
_lv = PYRAMID.build(to_root_level(load_cached(_r.image_id, 'train')))
fig, axes = plt.subplots(1, len(_lv), figsize=(3.4 * len(_lv), 3.8))
axes = np.atleast_1d(axes)
for k, (ax, (s, im)) in enumerate(zip(axes, _lv)):
    ax.imshow(im, cmap='gray'); ax.set_title(f'Level {k}  scale={s:.2f}  {im.shape[0]}x{im.shape[1]}', fontsize=9)
    ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')
fig.suptitle(f'Image pyramid ({len(_lv)} levels built; root window = {ROOT_CH}x{ROOT_CW} cells)')
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'image_pyramid.png', dpi=110); plt.show()

## Section 6 — Training Data Construction

**Positives.** For every training positive (no cap) we rescale the image so that the consensus heart box *plus margin* has the
same area as the root window, compute the block map **once**, and cut out the window centred on the heart. We add
shifted copies (±`POS_JITTER_CELLS` cells, shift only: **no horizontal flip**, because the heart is asymmetric).

**Negatives.** (a) `NEG_PER_IMAGE` random windows (random pyramid level, centre inside the position prior) from each of up to
`INIT_NEG_IMAGES` negative images; (b) random windows from the *positive* images that have IoU < `NEG_IOU_MAX` with the heart box.

**Why hard-negative mining (Section 7)?** Random negatives are mostly easy (empty lung, ribs). The classifier's real
mistakes are rare look-alikes: mediastinal contours, large vessels, effusions. After a first training round we scan negative
images densely, collect the windows the current model wrongly likes best, and retrain with them.

Level geometry helpers defined here (`build_levels`, `window_to_boxes`) are reused by the detector in Section 9.

In [ ]:
# ---------------- Shared level / window helpers ----------------
def compute_iou(boxA, boxB):
    """Intersection over union of two boxes (x1, y1, x2, y2). Returns 0.0 for degenerate boxes."""
    ix1, iy1 = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    ix2, iy2 = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    aA = max(0.0, boxA[2] - boxA[0]) * max(0.0, boxA[3] - boxA[1])
    aB = max(0.0, boxB[2] - boxB[0]) * max(0.0, boxB[3] - boxB[1])
    union = aA + aB - inter
    return float(inter / union) if union > 0 else 0.0

def build_levels(cache_img, hog=None, with_parts=False, timings=None):
    """Pyramid + ONE block map per level (root grid, and optionally the finer part grid).

    Each level dict has: scale, bm (root block map), pbm (part block map or None), kx, ky
    (cache pixels per level pixel, used to map windows back to cache space).
    """
    hog = hog or HOG
    t0 = time.perf_counter()
    root_img = to_root_level(cache_img)
    pyr = PYRAMID.build(root_img)
    part_imgs = []
    for _, im in pyr:
        if with_parts:   # part grid is exactly PART_RES_FACTOR times finer than the root grid of this level
            part_imgs.append(resize_to(cache_img, im.shape[0] * PART_RES_FACTOR, im.shape[1] * PART_RES_FACTOR))
    if timings is not None:
        timings['pyramid'] = timings.get('pyramid', 0.0) + (time.perf_counter() - t0) * 1e3
    t0 = time.perf_counter()
    levels = []
    for k, (s, im) in enumerate(pyr):
        levels.append({'scale': s, 'bm': hog.block_map(im),
                       'pbm': hog.block_map(part_imgs[k]) if with_parts else None,
                       'kx': CACHE_SIZE / im.shape[1], 'ky': CACHE_SIZE / im.shape[0]})
    if hog.device == 'cuda':
        torch.cuda.synchronize()
    if timings is not None:
        timings['hog'] = timings.get('hog', 0.0) + (time.perf_counter() - t0) * 1e3
    return levels

def window_to_boxes(level, cy, cx, ch=None, cw=None):
    """Window at top-left cell (cy, cx) of a level -> (window_box, heart_box) in CACHE space.

    window_box includes the context margin; heart_box is the window shrunk by 1/(1+2*margin) about its centre.
    """
    ch = ch or ROOT_CH; cw = cw or ROOT_CW
    x1, y1 = cx * CELL_W * level['kx'], cy * CELL_H * level['ky']
    x2, y2 = (cx + cw) * CELL_W * level['kx'], (cy + ch) * CELL_H * level['ky']
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    hw, hh = (x2 - x1) / (1 + 2 * ROOT_CONTEXT_MARGIN) / 2, (y2 - y1) / (1 + 2 * ROOT_CONTEXT_MARGIN) / 2
    return (x1, y1, x2, y2), (mx - hw, my - hh, mx + hw, my + hh)

def prior_ok(cx_cache, cy_cache):
    """True if a window centre (cache px) lies inside the position prior (or the prior is disabled)."""
    if not USE_POSITION_PRIOR:
        return True
    return PRIOR_RECT[0] <= cx_cache <= PRIOR_RECT[1] and PRIOR_RECT[2] <= cy_cache <= PRIOR_RECT[3]

def prior_mask_for_level(level, n_y, n_x, stride=STRIDE_CELLS):
    """Boolean (n_y, n_x) mask over window top-left cells whose CENTRE satisfies the prior."""
    if not USE_POSITION_PRIOR:
        return torch.ones(n_y, n_x, dtype=torch.bool)
    ys = (torch.arange(n_y) * stride + ROOT_CH / 2) * CELL_H * level['ky']
    xs = (torch.arange(n_x) * stride + ROOT_CW / 2) * CELL_W * level['kx']
    my = (ys >= PRIOR_RECT[2]) & (ys <= PRIOR_RECT[3])
    mx = (xs >= PRIOR_RECT[0]) & (xs <= PRIOR_RECT[1])
    return my[:, None] & mx[None, :]

PART_PAD_CELLS = ROOT_MAX_CELLS + POS_JITTER_CELLS + 2     # zero padding (root cells) around scaled positive images

def locate_positive(cache_img, box_cache, need_parts=False):
    """Rescale a positive image so box+margin has the root-window area; return its block map(s) and window cell.

    Returns dict(bm, cy, cx, pbm, pcy, pcx). (cy, cx) is the top-left cell of the window centred on the heart;
    (pcy, pcx) is the matching top-left cell on the finer part grid (PART_RES_FACTOR times finer).
    """
    x1, y1, x2, y2 = box_cache
    bw, bh = (x2 - x1) * (1 + 2 * ROOT_CONTEXT_MARGIN), (y2 - y1) * (1 + 2 * ROOT_CONTEXT_MARGIN)
    win_area = (ROOT_CH * CELL_H * DET_DOWNSCALE) * (ROOT_CW * CELL_W * DET_DOWNSCALE)   # window area in cache px
    s = math.sqrt(win_area / max(bw * bh, 1e-6))              # uniform scale: box+margin -> window area
    h = w = max(int(round(ROOT_LEVEL_SIZE * s)), (max(ROOT_CH, ROOT_CW) + 2) * CELL_H)   # square, like the cache
    root_img = resize_to(cache_img, h, w)
    pad = PART_PAD_CELLS * CELL_H
    root_img = np.pad(root_img, pad, constant_values=PAD_VALUE)
    ccx, ccy = (x1 + x2) / 2 / CACHE_SIZE * w + pad, (y1 + y2) / 2 / CACHE_SIZE * h + pad   # heart centre (level px)
    cy = int(round((ccy - ROOT_CH * CELL_H / 2) / CELL_H)); cx = int(round((ccx - ROOT_CW * CELL_W / 2) / CELL_W))
    out = {'bm': HOG.block_map(root_img), 'cy': cy, 'cx': cx, 'pbm': None, 'pcy': None, 'pcx': None}
    if need_parts:
        part_img = resize_to(cache_img, h * PART_RES_FACTOR, w * PART_RES_FACTOR)
        part_img = np.pad(part_img, pad * PART_RES_FACTOR, constant_values=PAD_VALUE)
        out.update(pbm=HOG.block_map(part_img), pcy=cy * PART_RES_FACTOR, pcx=cx * PART_RES_FACTOR)
    return out
print('helpers ready')

In [ ]:
# ---------------- build_feature_matrix ----------------
def _random_window(levels, rng, avoid_box=None):
    """Sample a random valid window (level k, cy, cx) with centre inside the prior (and IoU<NEG_IOU_MAX with avoid_box)."""
    for _ in range(NEG_TRIES):
        k = int(rng.integers(len(levels)))
        lv = levels[k]; nby, nbx = lv['bm'].shape[:2]
        ny, nx = nby - ROOT_BH + 1, nbx - ROOT_BW + 1
        if ny < 1 or nx < 1:
            continue
        cy, cx = int(rng.integers(ny)), int(rng.integers(nx))
        wbox, hbox = window_to_boxes(lv, cy, cx)
        if not prior_ok((wbox[0] + wbox[2]) / 2, (wbox[1] + wbox[3]) / 2):
            continue
        if avoid_box is not None and compute_iou(hbox, avoid_box) >= NEG_IOU_MAX:
            continue
        return k, cy, cx
    return None

def build_feature_matrix(rec, n_neg_images=INIT_NEG_IMAGES, seed=RANDOM_SEED):
    """Initial training matrix for the root filter (positives with jitter + random negatives).

    Returns X (float16 CPU tensor), y (numpy 0/1), ids (list of tags describing every row).
    """
    rng = np.random.default_rng(seed)
    feats, labels, ids = [], [], []
    pos = rec[rec.has_target == 1]; neg = rec[rec.has_target == 0]
    shifts = [(0, 0)] + [(dy, dx) for d in range(1, POS_JITTER_CELLS + 1)
                         for dy, dx in ((d, 0), (-d, 0), (0, d), (0, -d))]
    skipped = 0
    # ---- positives (ALL of them) + negatives from outside the heart box of positive images ----
    for r in tqdm(pos.itertuples(), total=len(pos), desc='positives'):
        img = load_cached(r.image_id, 'train')
        loc = locate_positive(img, (r.cx1, r.cy1, r.cx2, r.cy2))
        for dy, dx in shifts:
            v = HOG.window(loc['bm'], loc['cy'] + dy, loc['cx'] + dx, ROOT_CH, ROOT_CW)
            if v is None:
                skipped += 1; continue
            feats.append(v.to(STORE_DTYPE).cpu()); labels.append(1); ids.append(f'{r.image_id}|pos|{dy},{dx}')
        levels = build_levels(img)
        for _ in range(NEG_PER_IMAGE):
            w = _random_window(levels, rng, avoid_box=(r.cx1, r.cy1, r.cx2, r.cy2))
            if w is None:
                continue
            k, cy, cx = w
            feats.append(HOG.window(levels[k]['bm'], cy, cx, ROOT_CH, ROOT_CW).to(STORE_DTYPE).cpu())
            labels.append(0); ids.append(f'{r.image_id}|negwin|{k},{cy},{cx}')
    # ---- negatives from negative images ----
    neg_sel = neg if n_neg_images is None else neg.sample(min(n_neg_images, len(neg)), random_state=seed)
    for r in tqdm(neg_sel.itertuples(), total=len(neg_sel), desc='negatives'):
        levels = build_levels(load_cached(r.image_id, 'train'))
        for _ in range(NEG_PER_IMAGE):
            w = _random_window(levels, rng)
            if w is None:
                continue
            k, cy, cx = w
            feats.append(HOG.window(levels[k]['bm'], cy, cx, ROOT_CH, ROOT_CW).to(STORE_DTYPE).cpu())
            labels.append(0); ids.append(f'{r.image_id}|neg|{k},{cy},{cx}')
    X = torch.stack(feats); y = np.array(labels, dtype=np.int64)
    print(f'(skipped {skipped} jittered positive windows that left the padded image)')
    return X, y, ids

X0, y0, ids0 = build_feature_matrix(train_rec)
print(f'\nX.shape = {tuple(X0.shape)} | dtype = {X0.dtype} | memory = {X0.numel() * X0.element_size() / 1024**2:.0f} MB')
print(f'class balance: {int(y0.sum()):,} positive / {int((1 - y0).sum()):,} negative '
      f'({100 * y0.mean():.1f}% positive)')
_xf = X0[:2000].float()
print(pd.DataFrame({'mean': [_xf.mean().item()], 'std': [_xf.std().item()], 'min': [_xf.min().item()],
                    'max': [_xf.max().item()], 'feature dim': [X0.shape[1]]}).round(4).to_string(index=False))

## Section 7 — Root Filter Training (GPU, Hard-Negative Mining)

**The root filter** is a global template of how a cardiomegaly window looks in HOG space: a single weight vector `w`
(one weight per HOG feature of the window) plus a bias. The score of a window is `w · φ(window) + b`. We train it
**discriminatively** as a linear SVM (squared hinge loss + L2 penalty, class-weighted), so `w` is shaped to separate hearts from
everything else instead of just averaging heart pictures.

**Implementation.** The objective and its exact gradient are evaluated chunk by chunk on the GPU and minimised
with L-BFGS from PyTorch (no autograd graph, so memory stays small). Features are standardised with fixed
statistics; at inference the standardisation is folded into the weights so a whole pyramid level is scored with **one `conv2d`**.

**Hard-negative mining.** Round 0 trains on random negatives. Each later round densely scores *all* negative training images
across the pyramid, keeps the highest-scoring windows (the model's own false alarms), adds them and retrains. Easy mined
negatives are dropped again to keep the training set compact.

This section also defines the dense candidate search and `nms`, which the detector in Section 9 reuses.

In [ ]:
class TorchSVM:
    """Linear SVM (squared hinge + L2, class-balanced) trained with L-BFGS on DEVICE.

    Objective: 0.5*||w||^2 + C * sum_i cw_i * max(0, 1 - y_i (w.x_i + b))^2, with x standardised by fixed (mu, sigma).
    Optionally projects selected weights onto w >= 0 between L-BFGS restarts (used for deformation costs).
    """

    def __init__(self, C=SVM_C, max_iter=SVM_MAX_ITER, device=DEVICE, fit_intercept=True):
        """Store hyper-parameters."""
        self.C, self.max_iter, self.device, self.fit_intercept = C, max_iter, device, fit_intercept

    @staticmethod
    def feature_stats(X, device, chunk=SVM_CHUNK):
        """Per-feature mean and std of X computed chunk-wise (float32)."""
        n = X.shape[0]
        s1 = torch.zeros(X.shape[1], device=device); s2 = torch.zeros(X.shape[1], device=device)
        for i in range(0, n, chunk):
            xb = X[i:i + chunk].to(device).float()
            s1 += xb.sum(0); s2 += (xb * xb).sum(0)
        mu = s1 / n
        std = (s2 / n - mu * mu).clamp(min=0).sqrt()
        return mu, std.clamp(min=1e-3)

    def fit(self, X, y, mu, sigma, w_init=None, b_init=0.0, nonneg_idx=None):
        """Fit on (X, y in {0,1}). Returns (w, b, objective); w lives in STANDARDISED feature space."""
        dev = self.device
        Xd = X.to(dev)                                         # stays float16; converted per chunk
        yy = torch.as_tensor(y).float().to(dev) * 2 - 1
        n = len(yy); n_pos = int((yy > 0).sum()); n_neg = n - n_pos
        cw = torch.where(yy > 0, torch.tensor(n / (2 * max(n_pos, 1)), device=dev),
                                 torch.tensor(n / (2 * max(n_neg, 1)), device=dev))
        mu, sigma = mu.to(dev), sigma.to(dev)
        D = Xd.shape[1]
        w0 = torch.zeros(D, device=dev) if w_init is None else w_init.to(dev).clone()
        p = torch.nn.Parameter(torch.cat([w0, torch.tensor([float(b_init)], device=dev)]))
        nn_idx = None if nonneg_idx is None else torch.as_tensor(nonneg_idx, device=dev, dtype=torch.long)

        def objective():
            """Return the objective value and set p.grad analytically."""
            w, b = p.data[:-1], p.data[-1]
            loss = torch.zeros((), device=dev); gw = torch.zeros(D, device=dev); gb = torch.zeros((), device=dev)
            for i in range(0, n, SVM_CHUNK):
                xb = (Xd[i:i + SVM_CHUNK].float() - mu) / sigma
                yb, cb = yy[i:i + SVM_CHUNK], cw[i:i + SVM_CHUNK]
                viol = (1 - yb * (xb @ w + b)).clamp(min=0)
                loss += (cb * viol * viol).sum()
                coef = -2 * self.C * cb * viol * yb
                gw += xb.T @ coef; gb += coef.sum()
            g = torch.cat([w + gw, (gb if self.fit_intercept else torch.zeros_like(gb)).view(1)])
            p.grad = g
            return 0.5 * (w @ w) + self.C * loss

        rounds = SVM_PROJ_ROUNDS if nn_idx is not None else 1
        opt = torch.optim.LBFGS([p], lr=1.0, max_iter=max(self.max_iter // rounds, 1), history_size=10,
                                line_search_fn='strong_wolfe', tolerance_grad=1e-6, tolerance_change=1e-9)
        for _ in range(rounds):
            opt.step(objective)
            if nn_idx is not None:
                p.data[nn_idx] = p.data[nn_idx].clamp(min=0)    # projection onto w >= 0
        with torch.no_grad():
            obj = float(objective())
        w, b = p.data[:-1].clone(), float(p.data[-1])
        return w, b, obj

    def decision(self, X, w, b, mu, sigma):
        """Decision values for X (chunk-wise)."""
        out = []
        for i in range(0, X.shape[0], SVM_CHUNK):
            xb = (X[i:i + SVM_CHUNK].to(self.device).float() - mu.to(self.device)) / sigma.to(self.device)
            out.append(xb @ w.to(self.device) + b)
        return torch.cat(out)

class RootFilter:
    """Linear root template over HOG windows of ROOT_CH x ROOT_CW cells (trained as a linear SVM)."""

    def __init__(self, C=SVM_C, ch=None, cw=None, device=DEVICE):
        """Create an untrained filter for a ch x cw-cell window."""
        self.C, self.device = C, device
        self.ch, self.cw = ch or ROOT_CH, cw or ROOT_CW
        self.bh, self.bw = HOG.blocks_in(self.ch, self.cw)
        self.dim = HOG.feature_dim(self.ch, self.cw)
        self.w = self.mu = self.sigma = None
        self.b = 0.0
        self.history = []

    @torch.no_grad()
    def fit(self, X, y, restandardise=False):
        """Train (or warm-start retrain) on X (n, dim) and y in {0,1}; returns the final objective value."""
        svm = TorchSVM(self.C, SVM_MAX_ITER, self.device)
        if self.mu is None or restandardise:
            self.mu, self.sigma = TorchSVM.feature_stats(X, self.device)
        self.w, self.b, obj = svm.fit(X, y, self.mu, self.sigma, w_init=self.w, b_init=self.b)
        return obj

    @torch.no_grad()
    def decision_function(self, X):
        """Scores of feature rows X (n, dim): w_eff . x + b_eff."""
        return TorchSVM(self.C, device=self.device).decision(X, self.w, self.b, self.mu, self.sigma)

    def effective(self):
        """Standardisation folded in: score = x . w_eff + b_eff."""
        w_eff = self.w / self.sigma
        return w_eff, self.b - float((self.mu * w_eff).sum())

    @torch.no_grad()
    def score_map(self, block_map, stride=STRIDE_CELLS):
        """Dense scores of all windows of a pyramid level with ONE conv2d. Returns (ny, nx) or None if too small.

        Entry (i, j) is the score of the window whose top-left cell is (i*stride, j*stride).
        """
        if block_map.shape[0] < self.bh or block_map.shape[1] < self.bw:
            return None
        w_eff, b_eff = self.effective()
        k = w_eff.view(self.bh, self.bw, HOG_BLOCK_DIM).permute(2, 0, 1).unsqueeze(0)
        x = block_map.to(self.device).permute(2, 0, 1).unsqueeze(0)
        return F.conv2d(x, k, stride=stride)[0, 0] + b_eff

    def to(self, device):
        """Return a deep copy living on `device`."""
        new = copy.deepcopy(self)
        new.device = device
        for a in ('w', 'mu', 'sigma'):
            if getattr(new, a) is not None:
                setattr(new, a, getattr(new, a).to(device))
        return new

    def save(self, path):
        """Save with joblib; tensors are stored on the CPU."""
        joblib.dump({'C': self.C, 'ch': self.ch, 'cw': self.cw, 'b': self.b, 'w': self.w.cpu(),
                     'mu': self.mu.cpu(), 'sigma': self.sigma.cpu(), 'history': self.history}, path)

    @classmethod
    def load(cls, path, device=DEVICE):
        """Load a filter saved with save() onto `device`."""
        d = joblib.load(path)
        rf = cls(d['C'], d['ch'], d['cw'], device)
        rf.w, rf.mu, rf.sigma = d['w'].to(device), d['mu'].to(device), d['sigma'].to(device)
        rf.b, rf.history = d['b'], d.get('history', [])
        return rf

# ---------------- Sanity check against sklearn LinearSVC on a 2k subset ----------------
_idx = np.random.default_rng(RANDOM_SEED).choice(len(y0), min(SANITY_SUBSET, len(y0)), replace=False)
_Xs, _ys = X0[_idx], y0[_idx]
_tmp = RootFilter(); _tmp.fit(_Xs, _ys)
_mine_acc = float(((_tmp.decision_function(_Xs).cpu().numpy() > 0).astype(int) == _ys).mean())
_sc = StandardScaler().fit(_Xs.float().numpy())
_lin = LinearSVC(C=SVM_C, class_weight='balanced', max_iter=SVM_MAX_ITER * 10).fit(_sc.transform(_Xs.float().numpy()), _ys)
_lin_acc = float(_lin.score(_sc.transform(_Xs.float().numpy()), _ys))
print(f'Sanity check on {len(_ys)} rows: torch SVM accuracy = {_mine_acc:.3f} | sklearn LinearSVC accuracy = {_lin_acc:.3f} '
      f'| difference = {100 * abs(_mine_acc - _lin_acc):.1f} points')
if abs(_mine_acc - _lin_acc) > 0.05:
    print('WARNING: the two solvers disagree by more than a few points; inspect SVM_MAX_ITER / SVM_C.')
del _tmp, _Xs

In [ ]:
# ---------------- NMS and dense candidate search (reused by the detector) ----------------
def nms(detections, iou_threshold=IOU_NMS, score_key='dpm_score'):
    """Greedy non-maximum suppression. `detections` are dicts with a 'bbox' (x1,y1,x2,y2) and `score_key`."""
    order = sorted(detections, key=lambda d: -d[score_key])
    kept = []
    for d in order:
        if all(compute_iou(d['bbox'], k['bbox']) < iou_threshold for k in kept):
            kept.append(d)
    return kept

@torch.no_grad()
def dense_root_candidates(levels, root_filter, top_k=ROOT_TOP_K, stride=STRIDE_CELLS):
    """Score every window of every level with the root filter; return the best `top_k` local maxima.

    Each candidate dict: root_score, level, cy, cx, win (box incl. margin), bbox (heart box) - all in CACHE space.
    Windows whose centre is outside the position prior are ignored.
    """
    cands = []
    for k, lv in enumerate(levels):
        sm = root_filter.score_map(lv['bm'], stride)
        if sm is None:
            continue
        ny, nx = sm.shape
        s = sm.masked_fill(~prior_mask_for_level(lv, ny, nx, stride).to(sm.device), float('-inf'))
        if LOCAL_MAX_KERNEL > 1:                                   # keep only local peaks of the score map
            pooled = F.max_pool2d(s[None, None], LOCAL_MAX_KERNEL, 1, LOCAL_MAX_KERNEL // 2)[0, 0]
            s = s.masked_fill(s < pooled, float('-inf'))
        flat = s.flatten()
        vals, idx = torch.topk(flat, min(top_k, flat.numel()))
        ok = torch.isfinite(vals)
        for v, i in zip(vals[ok].tolist(), idx[ok].tolist()):
            cy, cx = (i // nx) * stride, (i % nx) * stride
            win, box = window_to_boxes(lv, cy, cx)
            cands.append({'root_score': v, 'level': k, 'cy': cy, 'cx': cx, 'win': win, 'bbox': box})
    cands.sort(key=lambda d: -d['root_score'])
    return cands[:top_k]

def mine_hard_negatives(root_filter, neg_rec, desc='mining'):
    """Dense-scan negative images; return (features [n, dim] STORE_DTYPE on CPU, keys) of the top false alarms."""
    feats, keys = [], []
    for r in tqdm(neg_rec.itertuples(), total=len(neg_rec), desc=desc):
        levels = build_levels(load_cached(r.image_id, 'train'))
        cands = dense_root_candidates(levels, root_filter, top_k=HARD_NEG_PER_IMAGE * 4)
        picked = [c for c in nms(cands, IOU_NMS, 'root_score') if c['root_score'] > HARD_NEG_MARGIN][:HARD_NEG_PER_IMAGE]
        for c in picked:
            v = HOG.window(levels[c['level']]['bm'], c['cy'], c['cx'], ROOT_CH, ROOT_CW)
            feats.append(v.to(STORE_DTYPE).cpu()); keys.append((r.image_id, c['level'], c['cy'], c['cx']))
    X = torch.stack(feats) if feats else torch.zeros(0, ROOT_DIM, dtype=STORE_DTYPE)
    return X, keys
print('nms / dense_root_candidates / mine_hard_negatives ready')

In [ ]:
# ---------------- Train the root filter with hard-negative mining ----------------
def _train_stats(rf, X, y):
    """Accuracy, sensitivity and specificity of rf on the current training matrix."""
    pred = (rf.decision_function(X).cpu().numpy() > 0).astype(int)
    return (float((pred == y).mean()), float(pred[y == 1].mean()), float(1 - pred[y == 0].mean()))

root_filter = RootFilter()
obj = root_filter.fit(X0, y0)
acc, sens, spec = _train_stats(root_filter, X0, y0)
rounds_log = [{'round': 0, 'positives': int(y0.sum()), 'negatives': int((1 - y0).sum()), 'hard added': 0,
               'hard kept': 0, 'train acc': acc, 'sens': sens, 'spec': spec, 'objective': obj}]

neg_pool = train_rec[train_rec.has_target == 0]
if MINE_NEG_IMAGES is not None:
    neg_pool = neg_pool.sample(min(MINE_NEG_IMAGES, len(neg_pool)), random_state=RANDOM_SEED)
hard_X = torch.zeros(0, ROOT_DIM, dtype=STORE_DTYPE); hard_keys = []

for rnd in range(1, HARD_NEG_ROUNDS + 1):
    # 1) drop previously mined negatives that are now easy (score below the margin)
    if len(hard_keys):
        s_old = root_filter.decision_function(hard_X).cpu().numpy()
        keep = s_old >= HARD_NEG_MARGIN
        hard_X, hard_keys = hard_X[torch.from_numpy(keep)], [k for k, f in zip(hard_keys, keep) if f]
    # 2) mine new false alarms over the whole pyramid of every negative image
    new_X, new_keys = mine_hard_negatives(root_filter, neg_pool, desc=f'round {rnd} mining')
    seen = set(hard_keys)
    fresh = [i for i, k in enumerate(new_keys) if k not in seen]
    if fresh:
        hard_X = torch.cat([hard_X, new_X[fresh]]); hard_keys += [new_keys[i] for i in fresh]
    # 3) retrain (warm start)
    X = torch.cat([X0, hard_X]); y = np.concatenate([y0, np.zeros(len(hard_X), dtype=np.int64)])
    obj = root_filter.fit(X, y)
    acc, sens, spec = _train_stats(root_filter, X, y)
    rounds_log.append({'round': rnd, 'positives': int(y.sum()), 'negatives': int((1 - y).sum()),
                       'hard added': len(fresh), 'hard kept': len(hard_keys),
                       'train acc': acc, 'sens': sens, 'spec': spec, 'objective': obj})

root_filter.history = rounds_log
root_filter.save(PKL_DIR / 'root_filter_Cardiomegaly.pkl')
print(pd.DataFrame(rounds_log).round(4).to_string(index=False))
print(f'\nfeature dim = {root_filter.dim:,} | saved to {PKL_DIR / "root_filter_Cardiomegaly.pkl"}')
root_filter = RootFilter.load(PKL_DIR / 'root_filter_Cardiomegaly.pkl')        # proves the pkl round-trips onto DEVICE

# ---------------- Plot the learned filter ----------------
_we, _ = root_filter.effective()
_wm = _we.view(root_filter.bh, root_filter.bw, HOG_BLOCK_DIM).cpu().numpy()
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
_signed = _wm.sum(-1); _lim = np.abs(_signed).max()
im0 = axes[0].imshow(_signed, cmap='RdBu_r', vmin=-_lim, vmax=_lim)
axes[0].set_title('Signed weight summed over orientations\n(red = evidence FOR cardiomegaly)'); plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(np.clip(_wm, 0, None).sum(-1), cmap='hot')
axes[1].set_title('Positive weights only (where the template wants edges)'); plt.colorbar(im1, ax=axes[1])
for ax in axes:
    ax.set_xlabel('block column (x)'); ax.set_ylabel('block row (y)')
fig.suptitle(f'Root filter weights ({root_filter.bh} x {root_filter.bw} blocks x {HOG_BLOCK_DIM})')
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'root_filter_weights.png', dpi=110); plt.show()

## Section 8 — Latent SVM with Learned Part Filters

**Four parts** describe the heart outline at **finer resolution** (`PART_RES_FACTOR`× the root HOG grid):
the two lateral borders, the superior contour (great-vessel region) and the inferior contour (diaphragmatic surface).
Radiological convention: **image-left is the patient's right**, so "Part 0" (image-left edge) is the patient's *right* cardiac border
and "Part 1" (image-right edge) is the patient's *left* border. The anchors are only **initial priors** (relative to the
heart box); each part may move by up to `PART_SEARCH_CELLS` part-cells and pays a quadratic **spring** cost.

**Score of a window**

```
score = w_root·φ_root + Σ_i  max_δ [ w_i·φ_part_i(anchor_i + δ) − d_i·φ_d(δ) ] + b
φ_d(δ) = [dx, dx², dy, dy²]
```

**Why "latent"?** The best displacement δ of every part is **not annotated**. Training therefore alternates
(coordinate descent): **E-step**: with the current weights, find each part's best δ for every training window;
**M-step**: with those δ fixed, the score is *linear* in one long vector `[φ_root, φ_part_1..N, −φ_d_1..N, 1]`, so we simply re-fit a linear SVM.
The SVM therefore learns the part filters **and** the deformation costs together. Quadratic deformation weights are projected
to be ≥ 0 (a spring cannot reward distance); they start at `DEFORM_INIT_QUAD`.

**Negatives** for this stage are the windows the root filter finds most confusing (hard negatives), and each one also gets
its best-scoring part placement, so the model learns not to be fooled by the max over displacements.

In [ ]:
def part_geometry():
    """Initial part windows in PART-LEVEL cells, relative to the top-left of the root window.

    Anchors are given relative to the heart BOX; the window also contains the context margin, so they are re-mapped.
    Returns a list of dicts with ax, ay (offset), pw, ph (size in cells) and dim (feature dimension).
    """
    chp, cwp = ROOT_CH * PART_RES_FACTOR, ROOT_CW * PART_RES_FACTOR
    m, e = ROOT_CONTEXT_MARGIN, 1 + 2 * ROOT_CONTEXT_MARGIN
    geo = []
    for (rx, ry, fw, fh) in PART_ANCHORS:
        pw = min(max(PART_MIN_CELLS, int(round(fw / e * cwp))), cwp)
        ph = min(max(PART_MIN_CELLS, int(round(fh / e * chp))), chp)
        ax = min(max(int(round((m + rx) / e * cwp)), 0), cwp - pw)
        ay = min(max(int(round((m + ry) / e * chp)), 0), chp - ph)
        geo.append({'ax': ax, 'ay': ay, 'pw': pw, 'ph': ph, 'dim': HOG.feature_dim(ph, pw)})
    return geo

def crop_blocks(bm, y0, x0, h, w):
    """Zero-padded crop bm[y0:y0+h, x0:x0+w] of a block map (windows near borders / displaced parts may leave it)."""
    out = torch.zeros(h, w, bm.shape[-1], dtype=bm.dtype, device=bm.device)
    ys, ye = max(y0, 0), min(y0 + h, bm.shape[0]); xs, xe = max(x0, 0), min(x0 + w, bm.shape[1])
    if ye > ys and xe > xs:
        out[ys - y0:ye - y0, xs - x0:xe - x0] = bm[ys:ye, xs:xe]
    return out

class LatentSVM:
    """Deformable part model: root filter + N_PARTS part filters + learned quadratic deformation costs."""

    def __init__(self, n_parts=N_PARTS, C=SVM_C, max_iter=LSVM_ITER, device=DEVICE):
        """Set up geometry and the layout of the long feature vector."""
        self.n_parts, self.C, self.max_iter, self.device = n_parts, C, max_iter, device
        self.D = PART_SEARCH_CELLS
        self.geo = part_geometry()
        self.chp, self.cwp = ROOT_CH * PART_RES_FACTOR, ROOT_CW * PART_RES_FACTOR
        self.Hc, self.Wc = self.chp - 1 + 2 * self.D, self.cwp - 1 + 2 * self.D      # crop size in blocks
        self.R = ROOT_DIM
        o = self.R
        self.part_sl = []
        for g in self.geo:
            self.part_sl.append(slice(o, o + g['dim'])); o += g['dim']
        self.def_sl = []
        for _ in self.geo:
            self.def_sl.append(slice(o, o + 4)); o += 4
        self.total_dim = o + 1                                  # + constant 1 (bias feature)
        self.root_sl = slice(0, self.R)
        g = torch.arange(-self.D, self.D + 1, dtype=torch.float32)
        dy, dx = torch.meshgrid(g, g, indexing='ij')            # grid index [ty, tx] <-> (dy, dx) = (ty-D, tx-D)
        self.phi_grid = torch.stack([dx, dx * dx, dy, dy * dy], -1)       # (2D+1, 2D+1, 4)
        self.w = self.mu = self.sigma = None
        self.history = []

    # ---------- weight handling ----------
    def _set_scaling(self, mu_hog):
        """mu/sigma: standardise HOG dims; deformation dims unscaled; bias feature scaled by LSVM_BIAS_SCALE."""
        self.mu = torch.zeros(self.total_dim, device=self.device)
        self.sigma = torch.ones(self.total_dim, device=self.device)
        self.mu[:self.def_sl[0].start], self.sigma[:self.def_sl[0].start] = mu_hog
        self.sigma[-1] = 1.0 / LSVM_BIAS_SCALE

    def _refresh(self):
        """Pre-compute effective (un-standardised) filters used by the fast convolution path."""
        w_eff = self.w / self.sigma
        self.root_eff = w_eff[self.root_sl]
        self.part_eff = [w_eff[sl].view(g['ph'] - 1, g['pw'] - 1, HOG_BLOCK_DIM).permute(2, 0, 1).unsqueeze(0).contiguous()
                         for sl, g in zip(self.part_sl, self.geo)]
        self.def_w = [self.w[sl] for sl in self.def_sl]          # (d_x, d_xx, d_y, d_yy) per part (sigma = 1)

    def nonneg_indices(self):
        """Indices of the quadratic deformation weights (dx^2 and dy^2 terms), constrained >= 0."""
        return [sl.start + k for sl in self.def_sl for k in (1, 3)]

    # ---------- crops of the part-level block map ----------
    def make_crop(self, pbm, pcy, pcx):
        """Part-grid crop (Hc, Wc, 36) around a window whose top-left PART cell is (pcy, pcx)."""
        return crop_blocks(pbm, pcy - self.D, pcx - self.D, self.Hc, self.Wc)

    # ---------- inference of the latent displacements ----------
    @torch.no_grad()
    def infer_parts(self, crops, use_deformation=True):
        """Best displacement of every part. crops: (N, Hc, Wc, 36) on self.device (float32).

        Returns (best_deltas (N, P, 2) as (dx, dy) in part cells, part_resp (N, P) filter responses at the best
        delta [no standardisation offset], deform_costs (N, P)).
        """
        N, D = crops.shape[0], self.D
        x = crops.permute(0, 3, 1, 2)
        deltas = torch.zeros(N, self.n_parts, 2, dtype=torch.long, device=crops.device)
        resp_best = torch.zeros(N, self.n_parts, device=crops.device)
        cost_best = torch.zeros(N, self.n_parts, device=crops.device)
        for i, g in enumerate(self.geo):
            resp = F.conv2d(x, self.part_eff[i])[:, 0]                      # (N, Hc-bh+1, Wc-bw+1)
            if not use_deformation:
                resp_best[:, i] = resp[:, D + g['ay'], D + g['ax']]
                continue
            sub = resp[:, g['ay']:g['ay'] + 2 * D + 1, g['ax']:g['ax'] + 2 * D + 1]        # (N, 2D+1, 2D+1)
            cost = (self.phi_grid.to(crops.device) * self.def_w[i]).sum(-1)               # (2D+1, 2D+1)
            score = sub - cost
            flat = score.reshape(N, -1).argmax(1)
            ty, tx = flat // (2 * D + 1), flat % (2 * D + 1)
            ar = torch.arange(N, device=crops.device)
            deltas[:, i, 0], deltas[:, i, 1] = tx - D, ty - D
            resp_best[:, i], cost_best[:, i] = sub[ar, ty, tx], cost[ty, tx]
        return deltas, resp_best, cost_best

    def gather_part_feats(self, crops, deltas):
        """Part features at the chosen displacements -> list of (N, dim_i) tensors."""
        N, D = crops.shape[0], self.D
        outs = []
        for i, g in enumerate(self.geo):
            out = torch.empty(N, g['dim'], dtype=crops.dtype, device=crops.device)
            code = (deltas[:, i, 1] + D) * (2 * D + 1) + (deltas[:, i, 0] + D)
            for c in code.unique().tolist():
                idx = (code == c).nonzero().squeeze(1)
                dy, dx = c // (2 * D + 1) - D, c % (2 * D + 1) - D
                y0, x0 = D + g['ay'] + dy, D + g['ax'] + dx
                out[idx] = crops[idx, y0:y0 + g['ph'] - 1, x0:x0 + g['pw'] - 1, :].reshape(len(idx), -1)
            outs.append(out)
        return outs

    def build_vector(self, root_feat, part_feats, deltas):
        """Concatenate [phi_root, phi_part_1..N, -phi_d_1..N, 1] -> (N, total_dim), so the SVM learns BOTH filters and costs."""
        N = root_feat.shape[0]
        dx, dy = deltas[..., 0].float(), deltas[..., 1].float()
        phi = torch.stack([dx, dx * dx, dy, dy * dy], -1)                  # (N, P, 4)
        parts = [root_feat.float()] + [p.float() for p in part_feats] + [-phi[:, i] for i in range(self.n_parts)]
        parts.append(torch.ones(N, 1, device=root_feat.device))
        return torch.cat(parts, 1)

    @torch.no_grad()
    def decompose(self, X):
        """Exact score decomposition of feature rows X: root, parts, deformation costs, bias, total."""
        z = (X - self.mu) / self.sigma * self.w                             # contribution of every feature
        return {'root': z[:, self.root_sl].sum(1),
                'parts': torch.stack([z[:, sl].sum(1) for sl in self.part_sl], 1),
                'deform_cost': torch.stack([-z[:, sl].sum(1) for sl in self.def_sl], 1),   # positive = penalty
                'bias': z[:, -1], 'total': z.sum(1)}

    @torch.no_grad()
    def score_candidates(self, root_feats, crops, use_deformation=True):
        """Full DPM scoring of N windows. Returns (decomposition dict, deltas (N, P, 2)). Deltas are 0 if use_deformation=False."""
        root_feats, crops = root_feats.to(self.device).float(), crops.to(self.device).float()
        deltas, _, _ = self.infer_parts(crops, use_deformation)
        X = self.build_vector(root_feats, self.gather_part_feats(crops, deltas), deltas)
        return self.decompose(X), deltas

    def score_window(self, root_feat, crop, use_deformation=True):
        """Score ONE window; returns dict(total, root, parts[list], deform_cost[list], bias, deltas) as Python numbers."""
        dec, deltas = self.score_candidates(root_feat.view(1, -1), crop.unsqueeze(0), use_deformation)
        return {'total': float(dec['total'][0]), 'root': float(dec['root'][0]), 'parts': dec['parts'][0].tolist(),
                'deform_cost': dec['deform_cost'][0].tolist(), 'bias': float(dec['bias'][0]),
                'deltas': [tuple(d) for d in deltas[0].tolist()]}

    # ---------- training ----------
    def build_pool(self, train_records, root_filter):
        """Training pool: root windows + part crops of ALL positives and of the root filter's hardest negatives."""
        roots, crops, labels = [], [], []
        pos = train_records[train_records.has_target == 1]; neg = train_records[train_records.has_target == 0]
        for r in tqdm(pos.itertuples(), total=len(pos), desc='LSVM positives'):
            loc = locate_positive(load_cached(r.image_id, 'train'), (r.cx1, r.cy1, r.cx2, r.cy2), need_parts=True)
            v = HOG.window(loc['bm'], loc['cy'], loc['cx'], ROOT_CH, ROOT_CW)
            if v is None:
                continue
            roots.append(v.to(STORE_DTYPE).cpu()); crops.append(self.make_crop(loc['pbm'], loc['pcy'], loc['pcx']).to(STORE_DTYPE).cpu())
            labels.append(1)
        neg_sel = neg if LSVM_NEG_IMAGES is None else neg.sample(min(LSVM_NEG_IMAGES, len(neg)), random_state=RANDOM_SEED)
        for r in tqdm(neg_sel.itertuples(), total=len(neg_sel), desc='LSVM hard negatives'):
            levels = build_levels(load_cached(r.image_id, 'train'), with_parts=True)
            cands = nms(dense_root_candidates(levels, root_filter, top_k=LSVM_NEG_PER_IMAGE * 4), IOU_NMS, 'root_score')
            for c in cands[:LSVM_NEG_PER_IMAGE]:
                lv = levels[c['level']]
                roots.append(HOG.window(lv['bm'], c['cy'], c['cx'], ROOT_CH, ROOT_CW).to(STORE_DTYPE).cpu())
                crops.append(self.make_crop(lv['pbm'], c['cy'] * PART_RES_FACTOR, c['cx'] * PART_RES_FACTOR).to(STORE_DTYPE).cpu())
                labels.append(0)
        return torch.stack(roots), torch.stack(crops), np.array(labels, dtype=np.int64)

    @torch.no_grad()
    def _e_step(self, crops, use_prev):
        """Best displacements for all pool samples (zeros before the first M-step)."""
        N = crops.shape[0]
        if not use_prev:
            return torch.zeros(N, self.n_parts, 2, dtype=torch.long)
        out = []
        for i in range(0, N, LSVM_BATCH):
            out.append(self.infer_parts(crops[i:i + LSVM_BATCH].to(self.device).float())[0].cpu())
        return torch.cat(out)

    @torch.no_grad()
    def _assemble(self, roots, crops, deltas):
        """Build the (N, total_dim) STORE_DTYPE matrix for the M-step, batch by batch."""
        N = roots.shape[0]
        X = torch.empty(N, self.total_dim, dtype=STORE_DTYPE, device=self.device)
        for i in range(0, N, LSVM_BATCH):
            cb = crops[i:i + LSVM_BATCH].to(self.device).float(); db = deltas[i:i + LSVM_BATCH].to(self.device)
            X[i:i + LSVM_BATCH] = self.build_vector(roots[i:i + LSVM_BATCH].to(self.device), self.gather_part_feats(cb, db), db).to(STORE_DTYPE)
        return X

    def fit(self, train_records, root_filter, hog=None):
        """Coordinate descent: LSVM_ITER x (E-step: best part displacements, M-step: SVM on the long vectors).

        (Images are read from the cache written by the Preprocessor in Section 3; `hog` defaults to the global extractor.)
        """
        roots, crops, y = self.build_pool(train_records, root_filter)
        self.n_pos, self.n_neg = int(y.sum()), int((1 - y).sum())
        svm = TorchSVM(self.C, SVM_MAX_ITER, self.device, fit_intercept=False)
        self.history = []
        for it in tqdm(range(self.max_iter), desc='LSVM iterations'):
            deltas = self._e_step(crops, use_prev=(it > 0))
            X = self._assemble(roots, crops, deltas)
            if it == 0:                                                  # standardisation fixed from iteration 0
                mu_h, sd_h = TorchSVM.feature_stats(X[:, :self.def_sl[0].start], self.device)
                self._set_scaling((mu_h, sd_h)); self.w = torch.zeros(self.total_dim, device=self.device)
            self.w, _, obj = svm.fit(X, y, self.mu, self.sigma, w_init=self.w, nonneg_idx=self.nonneg_indices())
            if it == 0:                                                  # deformation features were all zero: set the prior
                for sl in self.def_sl:
                    self.w[sl.start:sl.stop] = torch.tensor([0.0, DEFORM_INIT_QUAD, 0.0, DEFORM_INIT_QUAD], device=self.device)
            self._refresh()
            pos_d = deltas[torch.from_numpy(y == 1)].abs().float().mean((0, 2)) if self.n_pos else torch.zeros(self.n_parts)
            self.history.append({'iteration': it, 'objective': obj, 'mean |delta| pos (parts 0-3)': [round(float(v), 2) for v in pos_d]})
            del X
        return self

    # ---------- persistence ----------
    def to(self, device):
        """Return a deep copy living on `device`."""
        new = copy.deepcopy(self); new.device = device
        for a in ('w', 'mu', 'sigma'):
            setattr(new, a, getattr(new, a).to(device))
        new._refresh()
        return new

    def save(self, path):
        """Save with joblib (CPU tensors)."""
        joblib.dump({'w': self.w.cpu(), 'mu': self.mu.cpu(), 'sigma': self.sigma.cpu(), 'history': self.history,
                     'meta': {'ROOT_CH': ROOT_CH, 'ROOT_CW': ROOT_CW, 'D': self.D, 'PART_RES_FACTOR': PART_RES_FACTOR,
                              'n_parts': self.n_parts, 'C': self.C}}, path)

    @classmethod
    def load(cls, path, device=DEVICE):
        """Load a model saved with save() onto `device` (checks the geometry still matches the notebook constants)."""
        d = joblib.load(path); m = d['meta']
        assert (m['ROOT_CH'], m['ROOT_CW'], m['PART_RES_FACTOR'], m['D']) == (ROOT_CH, ROOT_CW, PART_RES_FACTOR, PART_SEARCH_CELLS), \
            'saved LSVM geometry differs from the current constants; retrain'
        obj = cls(m['n_parts'], m['C'], device=device)
        obj.w, obj.mu, obj.sigma, obj.history = d['w'].to(device), d['mu'].to(device), d['sigma'].to(device), d['history']
        obj._refresh()
        return obj

LSVM_BATCH = 256        # samples per batch in the E-step / assembly (memory control)
_lsvm_probe = LatentSVM()
print(pd.DataFrame([{'part': i, 'name': PART_NAMES[i], 'anchor (cells ay,ax)': (g['ay'], g['ax']),
                     'size (cells h,w)': (g['ph'], g['pw']), 'feature dim': g['dim']} for i, g in enumerate(_lsvm_probe.geo)]).to_string(index=False))
print(f'search window: +/-{PART_SEARCH_CELLS} part cells -> {(2 * PART_SEARCH_CELLS + 1) ** 2} displacements per part')
print(f'combined feature dimension = {_lsvm_probe.total_dim:,} (root {ROOT_DIM:,} + parts {sum(g["dim"] for g in _lsvm_probe.geo):,} '
      f'+ {4 * N_PARTS} deformation + 1 bias)')
del _lsvm_probe

In [ ]:
# ---------------- Train and save the latent SVM ----------------
lsvm = LatentSVM().fit(train_rec, root_filter)
lsvm.save(PKL_DIR / 'lsvm_Cardiomegaly.pkl')
lsvm = LatentSVM.load(PKL_DIR / 'lsvm_Cardiomegaly.pkl')       # proves the pkl round-trips onto DEVICE

print(f'\nCombined feature dimension : {lsvm.total_dim:,}')
print(f'Iterations run             : {len(lsvm.history)} (LSVM_ITER={LSVM_ITER})')
print(f'Training pool              : {lsvm.n_pos if hasattr(lsvm, "n_pos") else "n/a"} positives (see build_pool)')
print(pd.DataFrame(lsvm.history).to_string(index=False))
_dw = torch.stack(lsvm.def_w).cpu().numpy()
print('\nLearned deformation weights (d_x, d_xx, d_y, d_yy) per part:')
print(pd.DataFrame(_dw, columns=['d_x', 'd_xx', 'd_y', 'd_yy'], index=PART_NAMES).round(4).to_string())

## Section 9 — Dense Detection Engine

The detector is **two-stage**:

1. **Dense root scoring.** Build the pyramid, compute *one* HOG block map per level (plus one finer part-level map),
   and score *every* window of every level with a single convolution. Windows must be centred inside the position prior; we keep
   the best `ROOT_TOP_K` local maxima over all levels (a ranking, not a hard threshold) and run NMS on them → **root detections**.
2. **Part re-scoring.** Each candidate is re-scored with the full DPM: the best displacement of each part is found inside a
   small search window, and the exact score decomposition (root, parts, deformation, bias) is recorded. A final NMS gives the
   **DPM detections**.

The *image-level* score is the best detection score (DPM score, or root score in root-only mode); an image with no valid
window gets `NO_DET_SCORE`. `nms` and `compute_iou` were defined in Sections 6–7 so training could reuse them.
Setting `use_deformation=False` keeps the parts but freezes every displacement at 0 (used as an ablation in Section 12).

In [ ]:
def _sync(device):
    """Wait for queued GPU work so that wall-clock timings are honest."""
    if device == 'cuda':
        torch.cuda.synchronize()

def part_boxes(level, cy, cx, geo, deltas=None):
    """Part boxes in CACHE space for a window at top-left root cell (cy, cx). deltas: list of (dx, dy) part cells."""
    boxes = []
    for i, g in enumerate(geo):
        dx, dy = (0, 0) if deltas is None else deltas[i]
        py, px = cy * PART_RES_FACTOR + g['ay'] + dy, cx * PART_RES_FACTOR + g['ax'] + dx      # top-left part cell
        sx, sy = level['kx'] / PART_RES_FACTOR, level['ky'] / PART_RES_FACTOR                  # cache px per part-level px
        boxes.append((px * CELL_W * sx, py * CELL_H * sy, (px + g['pw']) * CELL_W * sx, (py + g['ph']) * CELL_H * sy))
    return boxes

@torch.no_grad()
def detect(img, root_filter, lsvm=None, use_deformation=True, levels=None, timings=None, hog=None):
    """Two-stage detector on ONE cache-space image (uint8, CACHE_SIZE x CACHE_SIZE).

    Returns dict: score (image-level), final (list of detections, best first), root_dets (after root NMS),
    n_raw (candidates before NMS), n_levels, mode. Detection dicts hold root_score, dpm_score, bbox, win, level, cy, cx,
    part_boxes, anchor_boxes, deltas, decomp. Boxes are in CACHE space. `timings` (dict) collects phase times in ms.
    """
    hog = hog or HOG
    dev = root_filter.device
    if levels is None:
        levels = build_levels(img, hog, with_parts=lsvm is not None, timings=timings)
    elif lsvm is not None and levels[0]['pbm'] is None:
        raise ValueError('levels were built without part maps but an LSVM was given')
    mode = 'root' if lsvm is None else ('dpm' if use_deformation else 'dpm_nodeform')

    _sync(dev); t0 = time.perf_counter()
    cands = dense_root_candidates(levels, root_filter)               # steps 2-3: dense scoring, prior, top-K local maxima
    _sync(dev)
    if timings is not None:
        timings['root_scoring'] = timings.get('root_scoring', 0.0) + (time.perf_counter() - t0) * 1e3
    out = {'score': NO_DET_SCORE, 'final': [], 'root_dets': [], 'n_raw': len(cands), 'n_levels': len(levels), 'mode': mode}
    if not cands:                                                     # edge case: no valid window at all
        return out

    t0 = time.perf_counter()
    root_dets = nms([dict(c, dpm_score=c['root_score']) for c in cands], IOU_NMS, 'root_score')   # step 4
    if timings is not None:
        timings['nms'] = timings.get('nms', 0.0) + (time.perf_counter() - t0) * 1e3
    out['root_dets'] = root_dets

    if lsvm is None:
        out['final'] = root_dets
    else:                                                             # steps 5-6: part rescoring + NMS
        _sync(dev); t0 = time.perf_counter()
        root_feats = torch.stack([hog.window(levels[c['level']]['bm'], c['cy'], c['cx'], ROOT_CH, ROOT_CW) for c in cands])
        crops = torch.stack([lsvm.make_crop(levels[c['level']]['pbm'], c['cy'] * PART_RES_FACTOR, c['cx'] * PART_RES_FACTOR)
                             for c in cands])
        dec, deltas = lsvm.score_candidates(root_feats, crops, use_deformation)
        _sync(dev)
        if timings is not None:
            timings['part_scoring'] = timings.get('part_scoring', 0.0) + (time.perf_counter() - t0) * 1e3
        tot, rt, pt = dec['total'].cpu().numpy(), dec['root'].cpu().numpy(), dec['parts'].cpu().numpy()
        dc, bs, dl = dec['deform_cost'].cpu().numpy(), dec['bias'].cpu().numpy(), deltas.cpu().numpy()
        scored = []
        for j, c in enumerate(cands):
            d_list = [(int(a), int(b)) for a, b in dl[j]]
            lv = levels[c['level']]
            scored.append(dict(c, dpm_score=float(tot[j]), deltas=d_list,
                               part_boxes=part_boxes(lv, c['cy'], c['cx'], lsvm.geo, d_list),
                               anchor_boxes=part_boxes(lv, c['cy'], c['cx'], lsvm.geo, None),
                               decomp={'root': float(rt[j]), 'parts': pt[j].tolist(), 'deform_cost': dc[j].tolist(),
                                       'bias': float(bs[j]), 'total': float(tot[j])}))
        t0 = time.perf_counter()
        out['final'] = nms(scored, IOU_NMS, 'dpm_score')
        if timings is not None:
            timings['nms'] = timings.get('nms', 0.0) + (time.perf_counter() - t0) * 1e3
    key = 'root_score' if lsvm is None else 'dpm_score'
    out['score'] = max(d[key] for d in out['final']) if out['final'] else NO_DET_SCORE
    return out

# ---------------- Smoke test on one validation positive ----------------
_r = val_rec[val_rec.has_target == 1].iloc[0]
_img = load_cached(_r.image_id, 'train')
_res = detect(_img, root_filter, lsvm)
print(f'mode={_res["mode"]} | levels={_res["n_levels"]} | raw candidates={_res["n_raw"]} | final detections={len(_res["final"])}')
if _res['final']:
    _d = _res['final'][0]
    print(f'best: root={_d["root_score"]:.3f} dpm={_d["dpm_score"]:.3f} bbox={tuple(round(v) for v in _d["bbox"])} '
          f'| gt={tuple(round(v) for v in (_r.cx1, _r.cy1, _r.cx2, _r.cy2))} | IoU={compute_iou(_d["bbox"], (_r.cx1, _r.cy1, _r.cx2, _r.cy2)):.2f}')
    print(f'decomposition sums to total: {_d["decomp"]["root"] + sum(_d["decomp"]["parts"]) - sum(_d["decomp"]["deform_cost"]) + _d["decomp"]["bias"]:.4f} '
          f'vs {_d["decomp"]["total"]:.4f}')

## Section 10 — Validation & Metrics

We run the detector on every validation image (natural class ratio, never used for training) and compare the image-level score with
the ground-truth label. Metrics:

- **Sensitivity / recall** = TP / (TP + FN): the fraction of sick patients we catch. In screening a **miss is far more costly than a false alarm**, so this usually matters more than accuracy.
- **Specificity** = TN / (TN + FP) and **FPR**: how many healthy patients are flagged. **Precision (PPV)** and **NPV** give the same information from the clinician's side.
- **Accuracy** is misleading when most images are negative (a model that never fires scores ~85 %).
- **F1 / F2** combine precision and recall (F2 weights recall higher); **MCC** is a balanced single number even for skewed classes.
- **ROC-AUC** is threshold-free ranking quality; **PR-AUC** focuses on the positive class and is more informative under class imbalance.
- **Localisation:** mean IoU of the top detection on true positives, and the **hit-rate** (share of positives whose best box has IoU ≥ 0.5 with the consensus box).

To save time, one sweep computes the root-only, no-deformation and full-DPM variants together (sharing the HOG pyramid). Section 10 reports the full DPM;
Section 12 compares all variants.

In [ ]:
HIT_IOU = 0.5                    # IoU needed for a localisation "hit"

def run_validation(rec, root_filter, lsvm, modes=('root', 'dpm_nodeform', 'dpm')):
    """Run detect() on every record for several model variants. Returns {mode: DataFrame}."""
    rows = {m: [] for m in modes}
    for r in tqdm(rec.itertuples(), total=len(rec), desc='validation'):
        img = load_cached(r.image_id, 'train')
        levels = build_levels(img, with_parts=True)          # HOG pyramid computed ONCE and shared by all variants
        gt = (r.cx1, r.cy1, r.cx2, r.cy2) if r.has_target else None
        for m in modes:
            res = detect(img, root_filter, None if m == 'root' else lsvm, use_deformation=(m != 'dpm_nodeform'), levels=levels)
            top = max(res['final'], key=lambda d: d['dpm_score']) if res['final'] else None
            rows[m].append({'image_id': r.image_id, 'gt_label': int(r.has_target), 'pred_score': res['score'],
                            'pred_label': int(res['score'] > THRESHOLD), 'pred_bbox': top['bbox'] if top else None,
                            'gt_bbox': gt, 'iou': compute_iou(top['bbox'], gt) if (top and gt) else (0.0 if gt else np.nan)})
    return {m: pd.DataFrame(v) for m, v in rows.items()}

VAL_RESULTS = run_validation(val_rec, root_filter, lsvm)
val_results_df = VAL_RESULTS['dpm']
print(val_results_df.drop(columns=['pred_bbox', 'gt_bbox']).head(8).round(3).to_string(index=False))

In [ ]:
def compute_metrics(y, score, thr=THRESHOLD, iou=None):
    """All Section-10 metrics as a dict. `iou` (per image, NaN for negatives) enables the localisation metrics."""
    y = np.asarray(y); score = np.asarray(score, dtype=float); pred = (score > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    div = lambda a, b: float(a / b) if b > 0 else float('nan')
    m = {'Accuracy': (tp + tn) / len(y), 'Precision/PPV': div(tp, tp + fp), 'Recall/Sensitivity': div(tp, tp + fn),
         'Specificity': div(tn, tn + fp), 'NPV': div(tn, tn + fn), 'F1': f1_score(y, pred, zero_division=0),
         'F2': fbeta_score(y, pred, beta=2, zero_division=0), 'MCC': matthews_corrcoef(y, pred),
         'ROC-AUC': roc_auc_score(y, score) if len(set(y)) > 1 else float('nan'),
         'PR-AUC': average_precision_score(y, score) if len(set(y)) > 1 else float('nan'),
         'Miss Rate': div(fn, tp + fn), 'FPR per image': div(fp, tn + fp)}
    if iou is not None:
        iou = np.asarray(iou, dtype=float)
        m['Mean IoU (true positives)'] = float(np.nanmean(iou[(y == 1) & (pred == 1)])) if ((y == 1) & (pred == 1)).any() else float('nan')
        m['Hit-rate@0.5 (positives)'] = float(np.mean(iou[y == 1] >= HIT_IOU)) if (y == 1).any() else float('nan')
    m['_counts'] = (int(tp), int(fp), int(tn), int(fn))
    return m

vm = compute_metrics(val_results_df.gt_label, val_results_df.pred_score, THRESHOLD, val_results_df.iou)
print(f'VALIDATION METRICS - DPM (threshold = {THRESHOLD}) | n = {len(val_results_df)}, positives = {int(val_results_df.gt_label.sum())}')
print(pd.DataFrame({'value': {k: v for k, v in vm.items() if k != '_counts'}}).round(4).to_string())
print('TP / FP / TN / FN =', vm['_counts'])

# Youden-optimal threshold: tuned ON the validation set, therefore OPTIMISTIC
_fpr, _tpr, _thr = roc_curve(val_results_df.gt_label, val_results_df.pred_score)
YOUDEN_THR = float(_thr[np.argmax(_tpr - _fpr)])
ym = compute_metrics(val_results_df.gt_label, val_results_df.pred_score, YOUDEN_THR, val_results_df.iou)
print(f'\nYouden-optimal threshold = {YOUDEN_THR:.4f}  [TUNED ON VALIDATION -> OPTIMISTIC, do not report as held-out performance]')
print(pd.DataFrame({'value': {k: ym[k] for k in ('Accuracy', 'Recall/Sensitivity', 'Specificity', 'F1', 'MCC')}}).round(4).to_string())

In [ ]:
# ---------------- Evaluation curves (2 x 2) ----------------
y_v, s_v = val_results_df.gt_label.values, val_results_df.pred_score.values
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fpr, tpr, _ = roc_curve(y_v, s_v)
axes[0, 0].plot(fpr, tpr, lw=2, label=f'DPM (AUC = {vm["ROC-AUC"]:.3f})'); axes[0, 0].plot([0, 1], [0, 1], 'k--', label='chance')
axes[0, 0].set(xlabel='False positive rate', ylabel='True positive rate (sensitivity)', title='ROC curve'); axes[0, 0].legend()
pr, rc, _ = precision_recall_curve(y_v, s_v)
axes[0, 1].plot(rc, pr, lw=2, label=f'DPM (AP = {vm["PR-AUC"]:.3f})'); axes[0, 1].axhline(y_v.mean(), color='k', ls='--', label='prevalence')
axes[0, 1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall curve'); axes[0, 1].legend()
tp_, fp_, tn_, fn_ = vm['_counts']; cm = np.array([[tn_, fp_], [fn_, tp_]])
axes[1, 0].imshow(cm, cmap='Blues')
for (i, j), v in np.ndenumerate(cm):
    axes[1, 0].text(j, i, str(v), ha='center', va='center', fontsize=16, color='black')
axes[1, 0].set(xticks=[0, 1], yticks=[0, 1], xticklabels=['pred neg', 'pred pos'], yticklabels=['true neg', 'true pos'],
               title=f'Confusion matrix (threshold {THRESHOLD})')
bins = np.linspace(min(s_v.min(), THRESHOLD) - 0.1, s_v.max() + 0.1, 40)
axes[1, 1].hist(s_v[(y_v == 1) & (s_v > THRESHOLD)], bins, color='green', alpha=.7, label='TP')
axes[1, 1].hist(s_v[(y_v == 0) & (s_v > THRESHOLD)], bins, color='red', alpha=.7, label='FP')
axes[1, 1].hist(s_v[s_v <= THRESHOLD], bins, color='gray', alpha=.4, label='below threshold')
axes[1, 1].axvline(THRESHOLD, color='k', ls='--', label='threshold')
axes[1, 1].set(xlabel='Image-level DPM score', ylabel='Number of images', title='Score histogram'); axes[1, 1].legend()
fig.suptitle(f'Validation evaluation - {TARGET_CLASS}')
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'evaluation_curves.png', dpi=110); plt.show()

# ---------------- Failure gallery: 4 FP + 4 FN ----------------
def show_failures(df, save_name, n=4):
    """Plot the n worst false positives (top row) and n worst false negatives (bottom row)."""
    fps = df[(df.gt_label == 0)].sort_values('pred_score', ascending=False).head(n)
    fns = df[(df.gt_label == 1)].sort_values('pred_score').head(n)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8.5))
    for row, (sub, tag) in zip(axes, ((fps, 'False positive'), (fns, 'False negative'))):
        for ax, (_, r) in zip(row, sub.iterrows()):
            ax.imshow(load_cached(r.image_id, 'train'), cmap='gray')
            if r.gt_bbox is not None:
                g = r.gt_bbox; ax.add_patch(mpatches.Rectangle((g[0], g[1]), g[2] - g[0], g[3] - g[1], fill=False, ec='lime', lw=2, label='ground truth'))
            if r.pred_bbox is not None:
                p = r.pred_bbox; ax.add_patch(mpatches.Rectangle((p[0], p[1]), p[2] - p[0], p[3] - p[1], fill=False, ec='red', lw=2, ls='--', label='top detection'))
            ax.set_title(f'{tag}: score {r.pred_score:.2f}', fontsize=10); ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')
    axes[0, 0].legend(loc='lower left', fontsize=8)
    fig.suptitle('Failure cases (green = consensus ground truth, red dashed = model)')
    plt.tight_layout(); plt.savefig(PLOTS_DIR / save_name, dpi=110); plt.show()

show_failures(val_results_df, 'failure_cases.png')

## Section 11 — Interpretability Visualisation

How a radiologist reads the figures:

1. **Preprocessed chest X-ray**: exactly what the model sees (aspect restored, contrast enhanced).
2. **HOG features**: the edge-orientation evidence the filters work on. Bright strokes follow the rib cage, diaphragm and heart border.
3. **Root response heat map**: how strongly the global heart template likes each location (green = looks like an enlarged heart, red = does not). A single compact blob on the heart is what we want.
4. **DPM detection**: the red box is the root detection; the four coloured dashed boxes are the parts after
   the latent search; white arrows go from each part's *anchor* (where the prior expects it) to where it was found. Long arrows mean the part had to move far and paid
   a larger deformation cost.

The bar chart splits the final score into root, part and deformation terms, and the part-activation grid shows the image
patch each part actually looked at.

In [ ]:
def build_response_map(img, root_filter):
    """Dense root response at the single root-level scale. Returns (map ndarray, extent=(x0, x1, y1, y0) in cache px)."""
    sm = root_filter.score_map(HOG.block_map(to_root_level(img)))
    if sm is None:
        return None, None
    m = sm.cpu().numpy(); ny, nx = m.shape; st = STRIDE_CELLS
    x0 = (ROOT_CW * CELL_W / 2 - CELL_W * st / 2) * DET_DOWNSCALE        # left edge of the first response pixel
    y0 = (ROOT_CH * CELL_H / 2 - CELL_H * st / 2) * DET_DOWNSCALE
    return m, (x0, x0 + nx * CELL_W * st * DET_DOWNSCALE, y0 + ny * CELL_H * st * DET_DOWNSCALE, y0)

def interpretability_figure(img, det, root_filter, save_path, suptitle=None):
    """4-panel interpretability figure for one detection dict (needs part_boxes, i.e. a DPM detection)."""
    fig, axes = plt.subplots(1, 4, figsize=(24, 6.5))
    axes[0].imshow(img, cmap='gray'); axes[0].set_title('1. Preprocessed CXR')
    axes[1].imshow(HOG.visualise(img), cmap='hot'); axes[1].set_title('2. Full-image HOG')
    rmap, ext = build_response_map(img, root_filter)
    axes[2].imshow(img, cmap='gray')
    if rmap is not None:
        hm = axes[2].imshow(rmap, cmap='RdYlGn', alpha=0.5, extent=ext); plt.colorbar(hm, ax=axes[2], fraction=0.046, label='root score')
    axes[2].set_title('3. Root response heat map')
    axes[3].imshow(img, cmap='gray')
    x1, y1, x2, y2 = det['bbox']
    handles = [mpatches.Patch(fill=False, ec='red', lw=3, label='Root (heart)')]
    axes[3].add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='red', lw=3))
    for i, (pb, ab) in enumerate(zip(det['part_boxes'], det['anchor_boxes'])):
        axes[3].add_patch(mpatches.Rectangle((pb[0], pb[1]), pb[2] - pb[0], pb[3] - pb[1], fill=False, ec=PART_COLORS[i], lw=2, ls='--'))
        axes[3].text(pb[0], pb[1] - 3, f'P{i}', color=PART_COLORS[i], fontsize=11, weight='bold')
        a, b = ((ab[0] + ab[2]) / 2, (ab[1] + ab[3]) / 2), ((pb[0] + pb[2]) / 2, (pb[1] + pb[3]) / 2)
        if abs(a[0] - b[0]) + abs(a[1] - b[1]) > 0:
            axes[3].annotate('', xy=b, xytext=a, arrowprops=dict(arrowstyle='->', color='white', lw=2))
        handles.append(mpatches.Patch(fill=False, ec=PART_COLORS[i], lw=2, ls='--', label=f'P{i} {PART_NAMES[i]}'))
    axes[3].text(8, CACHE_SIZE - 10, f'Root: {det["root_score"]:.2f} | DPM: {det["dpm_score"]:.2f}', color='yellow',
                 fontsize=12, weight='bold', bbox=dict(facecolor='black', alpha=0.6))
    axes[3].legend(handles=handles, loc='upper right', fontsize=7); axes[3].set_title('4. DPM detection (white arrows: anchor -> found position)')
    for ax in axes:
        ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')
    fig.suptitle(suptitle or f'Interpretable DPM - {TARGET_CLASS}', fontsize=15)
    plt.tight_layout(); plt.savefig(save_path, dpi=110); plt.show()

def plot_decomposition(det, save_path):
    """Bar chart: root, parts 0-3, bias, total deformation (red, negative) and the final score (bold)."""
    d = det['decomp']
    labels = ['Root'] + [f'Part {i}' for i in range(N_PARTS)] + ['Deformation', 'Bias', 'FINAL']
    vals = [d['root']] + list(d['parts']) + [-sum(d['deform_cost']), d['bias'], d['total']]
    colors = ['tab:blue'] + PART_COLORS + ['red', 'gray', 'black']
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(labels, vals, color=colors, edgecolor='black')
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v, f'{v:+.2f}', ha='center', va='bottom' if v >= 0 else 'top', fontsize=10)
    bars[-1].set_linewidth(3); ax.axhline(0, color='k', lw=0.8)
    ax.set_ylabel('Score contribution'); ax.set_xlabel('Model term'); ax.set_title('DPM score decomposition')
    plt.tight_layout(); plt.savefig(save_path, dpi=110); plt.show()

def plot_part_activations(img, det, save_path):
    """1 x N_PARTS grid: image patch of each part (after displacement) with its HOG overlaid."""
    fig, axes = plt.subplots(1, N_PARTS, figsize=(4.5 * N_PARTS, 5))
    for i, ax in enumerate(np.atleast_1d(axes)):
        x1, y1, x2, y2 = [int(round(v)) for v in det['part_boxes'][i]]
        x1, y1, x2, y2 = max(x1, 0), max(y1, 0), min(x2, CACHE_SIZE), min(y2, CACHE_SIZE)
        ax.set_title(f'P{i} {PART_NAMES[i]}\ncost={det["decomp"]["deform_cost"][i]:+.3f} | activation={det["decomp"]["parts"][i]:+.2f}', fontsize=9)
        if x2 - x1 < CELL_W * 2 or y2 - y1 < CELL_H * 2:
            ax.text(0.5, 0.5, 'part outside image', ha='center'); continue
        crop = img[y1:y2, x1:x2]
        big = resize_to(crop, max(crop.shape[0], CELL_H * 8), max(crop.shape[1], CELL_W * 8))
        ax.imshow(big, cmap='gray'); ax.imshow(HOG.visualise(big), cmap='hot', alpha=0.5)
        ax.set_xlabel('x (px, patch)'); ax.set_ylabel('y (px, patch)')
    fig.suptitle('Part activations (patch + HOG overlay)')
    plt.tight_layout(); plt.savefig(save_path, dpi=110); plt.show()

# ---------------- Top-scoring true positive ----------------
_tp = val_results_df[(val_results_df.gt_label == 1) & (val_results_df.pred_label == 1)]
if _tp.empty:
    print('No true positives on validation; using the highest-scoring positive instead.')
    _tp = val_results_df[val_results_df.gt_label == 1]
TOP_TP_ID = _tp.sort_values('pred_score', ascending=False).iloc[0].image_id
TOP_TP_IMG = load_cached(TOP_TP_ID, 'train')
TOP_TP_RES = detect(TOP_TP_IMG, root_filter, lsvm)
TOP_TP_DET = max(TOP_TP_RES['final'], key=lambda d: d['dpm_score'])
print(f'Image {TOP_TP_ID}: DPM score {TOP_TP_DET["dpm_score"]:.3f}')
interpretability_figure(TOP_TP_IMG, TOP_TP_DET, root_filter, PLOTS_DIR / 'interpretability_4panel.png')
plot_decomposition(TOP_TP_DET, PLOTS_DIR / 'score_decomposition.png')
plot_part_activations(TOP_TP_IMG, TOP_TP_DET, PLOTS_DIR / 'part_activations.png')

## Section 12 — Baselines & Ablations

Interpretability has a price: a part-based model is more complex than a single template and far more complex than a plain
classifier. We only accept that price if the extra structure *earns* its keep, so we compare four systems on the same validation images:

1. **Baseline 1 — global HOG + LinearSVC (image level):** HOG of the whole image (resized to `B1_SIZE`), one label per image, trained on the
   **train images** (not patches). It cannot localise, so IoU does not apply. It shows what a plain image classifier on the same features achieves.
2. **Baseline 2 — root only:** the Section 7 filter with the same dense pyramid and NMS, no parts. It isolates the value of *detection with a template*.
3. **Ablation — DPM without deformation:** parts are used but every displacement is fixed at 0. It isolates the value of the **latent search**.
4. **DPM (full):** root + parts + learned deformation.

In [ ]:
# ---------------- Baseline 1: image-level global HOG + LinearSVC ----------------
def global_hog_features(rec, batch=64):
    """Whole-image HOG vectors (n, dim): cached image -> B1_SIZE x B1_SIZE -> block map -> flatten."""
    out = []
    for i in tqdm(range(0, len(rec), batch), desc='global HOG'):
        ids = rec.image_id.values[i:i + batch]
        imgs = np.stack([resize_to(load_cached(j, 'train'), B1_SIZE, B1_SIZE) for j in ids])
        bm = HOG.block_maps(imgs)
        out.append(bm.reshape(bm.shape[0], -1).cpu().numpy())
    return np.concatenate(out)

Xb_tr, Xb_va = global_hog_features(train_rec), global_hog_features(val_rec)
sc_b1 = StandardScaler().fit(Xb_tr)
b1 = LinearSVC(C=B1_C, class_weight='balanced', dual=False, max_iter=2000).fit(sc_b1.transform(Xb_tr), train_rec.has_target.values)
b1_scores = b1.decision_function(sc_b1.transform(Xb_va))
print(f'Baseline 1: feature dim = {Xb_tr.shape[1]:,}, trained on {len(Xb_tr):,} IMAGES (image-level labels)')
del Xb_tr

# ---------------- Comparison table ----------------
y_val = val_rec.has_target.values
res = {'Baseline 1: global HOG': compute_metrics(y_val, b1_scores, 0.0, None),
       'Baseline 2: root only': compute_metrics(y_val, VAL_RESULTS['root'].pred_score, THRESHOLD, VAL_RESULTS['root'].iou),
       'Ablation: DPM, no deform.': compute_metrics(y_val, VAL_RESULTS['dpm_nodeform'].pred_score, THRESHOLD, VAL_RESULTS['dpm_nodeform'].iou),
       'DPM (full)': compute_metrics(y_val, VAL_RESULTS['dpm'].pred_score, THRESHOLD, VAL_RESULTS['dpm'].iou)}
rows = [('Accuracy', 'Accuracy'), ('Sensitivity', 'Recall/Sensitivity'), ('Specificity', 'Specificity'), ('F1', 'F1'),
        ('ROC-AUC', 'ROC-AUC'), ('PR-AUC', 'PR-AUC'), ('Mean IoU', 'Mean IoU (true positives)'), ('Hit-rate@0.5', 'Hit-rate@0.5 (positives)')]
comp = pd.DataFrame({name: {label: res[name].get(key, float('nan')) for label, key in rows} for name in res})
print(comp.round(3).fillna('N/A').to_string())
print('(Baseline 1 is image-level only: IoU / hit-rate are not applicable.)')

# ---------------- Grouped bar chart ----------------
fig, ax = plt.subplots(figsize=(11, 5.5))
metrics_plot = ['F1', 'ROC-AUC', 'Sensitivity']; wbar = 0.2; x = np.arange(len(metrics_plot))
for k, name in enumerate(comp.columns):
    vals = comp.loc[metrics_plot, name].values.astype(float)
    bars = ax.bar(x + (k - 1.5) * wbar, vals, wbar, label=name)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f'{v:.2f}', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(metrics_plot); ax.set_ylim(0, 1.05)
ax.set_xlabel('Metric'); ax.set_ylabel('Score'); ax.set_title('Baselines vs DPM (validation set)'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'baseline_comparison.png', dpi=120); plt.show()

# ---------------- Honest, automatic verdict ----------------
d_auc = comp.loc['ROC-AUC', 'DPM (full)'] - comp.loc['ROC-AUC', 'Baseline 2: root only']
d_f1 = comp.loc['F1', 'DPM (full)'] - comp.loc['F1', 'Baseline 2: root only']
d_def = comp.loc['ROC-AUC', 'DPM (full)'] - comp.loc['ROC-AUC', 'Ablation: DPM, no deform.']
d_glob = comp.loc['ROC-AUC', 'DPM (full)'] - comp.loc['ROC-AUC', 'Baseline 1: global HOG']
print(f'\nDPM vs root-only        : ROC-AUC {d_auc:+.3f}, F1 {d_f1:+.3f}  -> parts {"HELP" if d_auc > 0.005 else "do NOT clearly help"}')
print(f'DPM vs no-deformation   : ROC-AUC {d_def:+.3f}  -> latent displacement search {"HELPS" if d_def > 0.005 else "does NOT clearly help"}')
print(f'DPM vs global image HOG : ROC-AUC {d_glob:+.3f}  -> {"detector beats" if d_glob > 0.005 else "image-level classifier matches or beats"} the plain classifier')

### How to read the comparison (honest interpretation)

- **Parts vs root only.** If the full DPM's ROC-AUC / F1 are clearly above the root-only baseline, the part filters add real information (the borders
  and contours carry evidence the single template misses). If the gap is within noise (a few thousandths of AUC), then **the parts do not earn their complexity**
  for this problem and the root template is the better engineering choice; say so in any write-up.
- **Deformation vs no deformation.** A gain here means letting parts move (with a spring penalty) helps with the large natural variation in heart shape.
  No gain means the parts are acting as extra fixed templates.
- **Detector vs global classifier.** Cardiomegaly is a *global* property (heart width relative to chest width). A whole-image HOG classifier sees the global layout and
  may match or beat a local detector on image-level AUC, but it cannot say *where* the heart is and offers no part-level explanation. That is the interpretability trade-off.
- Differences of about 0.01 AUC on a validation set of this size are within sampling noise; do not over-interpret small gaps.

## Section 13 — Computational Profile

A pre-screening tool is useful in a clinic only if it is fast and small enough to run on modest hardware (a workstation next to the scanner, or an edge device)
without sending images off-site. We time every phase of inference on `BENCH_IMAGES` random validation PNGs (including PNG decoding and preprocessing),
measure peak host RAM and peak GPU memory, repeat the timing on the CPU for a few images, and report model size and estimated FLOPs.
GPU timings use `torch.cuda.synchronize()` around each phase. Peak host RAM is the process resident set size sampled after each image
(`tracemalloc` cannot see tensor memory).

In [ ]:
PHASES = ['preprocess', 'pyramid', 'hog', 'root_scoring', 'nms', 'part_scoring', 'total']

def benchmark(rows, root_filter, lsvm, hog, device, warmup=True):
    """Time every inference phase on the given records. Returns (DataFrame of ms per image, peak RSS MB, peak GPU MB)."""
    proc = psutil.Process(); peak_rss = proc.memory_info().rss
    def one(r):
        t = {}
        _sync(device); t0 = time.perf_counter()
        img, _ = PRE.preprocess(r.png_path, r.H, r.W)
        _sync(device); t['preprocess'] = (time.perf_counter() - t0) * 1e3
        detect(img, root_filter, lsvm, timings=t, hog=hog)
        _sync(device); t['total'] = (time.perf_counter() - t0) * 1e3
        return t
    first = next(rows.itertuples())
    if warmup:
        one(first)                                                  # warm-up (CUDA kernels, allocator)
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    out = []
    for r in tqdm(rows.itertuples(), total=len(rows), desc=f'benchmark ({device})'):
        out.append(one(r)); peak_rss = max(peak_rss, proc.memory_info().rss)
    peak_gpu = torch.cuda.max_memory_allocated() / 1024**2 if device == 'cuda' else float('nan')
    return pd.DataFrame(out).reindex(columns=PHASES).fillna(0.0), peak_rss / 1024**2, peak_gpu

bench_rows = val_rec.sample(min(BENCH_IMAGES, len(val_rec)), random_state=RANDOM_SEED)
bt, PEAK_RSS_MB, PEAK_GPU_MB = benchmark(bench_rows, root_filter, lsvm, HOG, DEVICE)
print(f'\nPER-IMAGE TIMING on {DEVICE.upper()} ({len(bt)} images, ms)')
print(pd.DataFrame({'mean': bt.mean(), 'std': bt.std()}).round(2).to_string())
print(f'Peak host RAM (process RSS): {PEAK_RSS_MB:.0f} MB | Peak GPU memory (tensors): '
      f'{"n/a (CPU run)" if DEVICE == "cpu" else f"{PEAK_GPU_MB:.0f} MB"}')

# ---- CPU-only timing on a small subset for comparison ----
hog_cpu, rf_cpu, lsvm_cpu = HOGExtractor(device='cpu'), root_filter.to('cpu'), lsvm.to('cpu')
bt_cpu, _, _ = benchmark(bench_rows.head(BENCH_CPU_IMAGES), rf_cpu, lsvm_cpu, hog_cpu, 'cpu', warmup=False)
cmp = pd.DataFrame({f'{DEVICE} mean ms': bt.mean(), 'CPU-only mean ms': bt_cpu.mean()})
cmp['CPU / ' + DEVICE] = cmp['CPU-only mean ms'] / cmp[f'{DEVICE} mean ms'].replace(0, np.nan)
print(f'\nCPU-ONLY COMPARISON ({len(bt_cpu)} images)'); print(cmp.round(2).to_string())
del hog_cpu, rf_cpu, lsvm_cpu

In [ ]:
# ---------------- Model size & FLOPs ----------------
def kb(p):
    """File size in KB."""
    return os.path.getsize(p) / 1024

size_rows = {'root_filter_Cardiomegaly.pkl': kb(PKL_DIR / 'root_filter_Cardiomegaly.pkl'),
             'lsvm_Cardiomegaly.pkl': kb(PKL_DIR / 'lsvm_Cardiomegaly.pkl')}
MODEL_SIZE_KB = sum(size_rows.values())

def estimate_flops(example_img, K=ROOT_TOP_K):
    """Multiply-add FLOPs per image from the ACTUAL filter shapes x windows x levels (HOG extraction not counted)."""
    lv = build_levels(example_img, with_parts=True)
    n_win = [max(l['bm'].shape[0] - ROOT_BH + 1, 0) * max(l['bm'].shape[1] - ROOT_BW + 1, 0) for l in lv]
    root = 2 * ROOT_DIM * sum(n_win)
    parts = 0
    for g in lsvm.geo:
        out_h, out_w = lsvm.Hc - (g['ph'] - 1) + 1, lsvm.Wc - (g['pw'] - 1) + 1       # conv positions per candidate crop
        parts += 2 * g['dim'] * out_h * out_w * K
    return root, parts, n_win

_ex = load_cached(bench_rows.image_id.iloc[0], 'train')
FLOPS_ROOT, FLOPS_PARTS, n_win = estimate_flops(_ex)
FLOPS_TOTAL = FLOPS_ROOT + FLOPS_PARTS
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU only'
print(pd.DataFrame({'size (KB)': size_rows}).round(1).to_string()); print(f'TOTAL model size: {MODEL_SIZE_KB:.1f} KB')
print(f'\nWindows per pyramid level: {n_win} (total {sum(n_win):,}) | root filter dim {ROOT_DIM:,}')
print(f'Estimated FLOPs per image: root scoring {FLOPS_ROOT / 1e6:.1f} M | part scoring ({ROOT_TOP_K} candidates) {FLOPS_PARTS / 1e6:.1f} M '
      f'| total {FLOPS_TOTAL / 1e6:.1f} MFLOPs')
print(f'Hardware used: {GPU_NAME}')
BENCH_STATS = {'timing_mean': bt.mean().to_dict(), 'peak_rss_mb': PEAK_RSS_MB, 'peak_gpu_mb': PEAK_GPU_MB,
               'model_kb': MODEL_SIZE_KB, 'flops': FLOPS_TOTAL, 'gpu': GPU_NAME}

## Section 14 — Full Structured Inference Report

This section runs the complete pipeline on **one test PNG** (the first file in `TEST_DIR`, or `TEST_IMAGE_PATH` if you set it):
preprocessing → HOG block maps → pyramid → dense root scoring → NMS → part re-scoring and score decomposition → response map →
interpretability figure, with timings and peak memory. It then writes a structured, human-readable report to `output/reports/`.
The test set has no labels; this is a demonstration of the deployed pipeline, not an evaluation.
Original image sizes come from `test_meta.csv`; without them the aspect ratio cannot be restored, so a validation image is used instead (and the report says so).

In [ ]:
# ---------------- Pick a test image ----------------
REPORT_NOTES = []
test_path, test_H, test_W, REPORT_SOURCE = None, None, None, 'test'
if TEST_IMAGE_PATH is not None:
    test_path = Path(TEST_IMAGE_PATH)
elif TEST_DIR.exists():
    _files = sorted(TEST_DIR.glob('*.png'))
    test_path = _files[0] if _files else None

if test_path is not None and test_path.exists() and TEST_META_PATH.exists():
    _m = pd.read_csv(TEST_META_PATH).set_index('image_id')
    if test_path.stem in _m.index:
        test_H, test_W = int(_m.loc[test_path.stem, 'dim0']), int(_m.loc[test_path.stem, 'dim1'])
if test_H is None:                                          # fall back to a validation image and SAY SO
    _fb = val_rec.iloc[0]
    test_path, test_H, test_W, REPORT_SOURCE = Path(_fb.png_path), int(_fb.H), int(_fb.W), 'validation (fallback)'
    REPORT_NOTES.append('No usable test image / test_meta.csv row was found, so a VALIDATION image was used. '
                        'Aspect restoration needs the original size, which is only known for images with a meta row.')
    print('NOTE:', REPORT_NOTES[-1])
print(f'Report image: {test_path.name} ({REPORT_SOURCE}) | original size H x W = {test_H} x {test_W}')

In [ ]:
# ---------------- Run the full pipeline with timings and peak memory ----------------
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
proc = psutil.Process(); rss_peak = proc.memory_info().rss
T = {}                                                      # phase -> ms
_sync(DEVICE); t_all = time.perf_counter()

t0 = time.perf_counter()                                    # 1. preprocess
rep_img, rep_meta = PRE.preprocess(test_path, test_H, test_W)
T['preprocess'] = (time.perf_counter() - t0) * 1e3

rep_levels = build_levels(rep_img, HOG, with_parts=True, timings=T)       # 2. HOG block maps + 3. pyramid (timed inside)
rss_peak = max(rss_peak, proc.memory_info().rss)
rep_res = detect(rep_img, root_filter, lsvm, levels=rep_levels, timings=T)  # 4. dense root, 5. NMS, 6. part rescoring
rss_peak = max(rss_peak, proc.memory_info().rss)

_sync(DEVICE); t0 = time.perf_counter()                     # 7. response map
rep_map, _ = build_response_map(rep_img, root_filter)
_sync(DEVICE); T['response_map'] = (time.perf_counter() - t0) * 1e3

rep_det = max(rep_res['final'], key=lambda d: d['dpm_score']) if rep_res['final'] else None
FIG_PATHS = {}
t0 = time.perf_counter()                                    # 8. interpretability figures
if rep_det is not None:
    FIG_PATHS['interp'] = PLOTS_DIR / f'report_{REPORT_ID}_interpretability.png'
    FIG_PATHS['decomp'] = PLOTS_DIR / f'report_{REPORT_ID}_score_decomposition.png'
    FIG_PATHS['parts'] = PLOTS_DIR / f'report_{REPORT_ID}_part_activations.png'
    interpretability_figure(rep_img, rep_det, root_filter, FIG_PATHS['interp'], f'Interpretable DPM - {TARGET_CLASS} ({test_path.name[:12]})')
    plot_decomposition(rep_det, FIG_PATHS['decomp'])
    plot_part_activations(rep_img, rep_det, FIG_PATHS['parts'])
else:
    print('No detections for this image: figures skipped (an empty detection list means "normal").')
T['figures'] = (time.perf_counter() - t0) * 1e3
_sync(DEVICE); T['total_pipeline'] = (time.perf_counter() - t_all) * 1e3
rss_peak = max(rss_peak, proc.memory_info().rss)
REPORT_PEAK_RSS_MB = rss_peak / 1024**2
REPORT_PEAK_GPU_MB = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE == 'cuda' else float('nan')
print(pd.DataFrame({'ms': T}).round(2).to_string())

In [ ]:
# ---------------- Generate, print and save the report ----------------
LINE = '=' * 71
SEP = '-' * 62
def bar(frac):
    """ASCII bar for a fraction in [0, 1]."""
    n = int(round(min(max(frac, 0.0), 1.0) * ASCII_BAR_WIDTH))
    return '#' * n + '.' * (ASCII_BAR_WIDTH - n)

def build_report():
    """Assemble all report blocks into one string."""
    L = []
    has_det = rep_det is not None
    score = rep_det['dpm_score'] if has_det else NO_DET_SCORE
    flagged = has_det and score > THRESHOLD
    conf = 'HIGH' if score >= CONF_HIGH else ('MODERATE' if score >= THRESHOLD else 'LOW')
    priority = 'URGENT' if (flagged and conf == 'HIGH') else ('ROUTINE' if flagged else 'LOW')
    hw = GPU_NAME if DEVICE == 'cuda' else f'{platform.processor() or platform.machine()} (CPU)'

    # Block 0 - header
    L += [LINE, '        MEDICAL ANOMALY DETECTION - INFERENCE REPORT', '               Part-Based Deformable Model (DPM)', LINE,
          f'Report ID        : {REPORT_ID}', f'Image File (PNG)  : {test_path.name}  [{REPORT_SOURCE}]',
          f'Original Size    : {test_H} x {test_W} (H x W)', f'Working Size     : {CACHE_SIZE} x {CACHE_SIZE} (aspect restored, letterboxed); root level {ROOT_LEVEL_SIZE} px',
          f'Device           : {DEVICE}', 'Scan Type        : Chest X-Ray', f'Target Class     : {TARGET_CLASS}',
          f'Date Processed   : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', f'Model Version    : {MODEL_VERSION}',
          'Dataset          : VinBigData Chest X-Ray Abnormalities Detection', LINE]
    for n in REPORT_NOTES:
        L.append(f'NOTE: {n}')

    # Block 1 - detection summary
    L += ['', 'BLOCK 1 - DETECTION SUMMARY', SEP]
    L.append(f'Anomaly Detected   : {"YES" if flagged else "NO"}')
    L.append(f'Anomaly Type       : {TARGET_CLASS if flagged else "None"}')
    L.append(f'Confidence Score   : {score:+.4f}')
    L.append(f'Detection Status   : {"ANOMALY FLAGGED" if flagged else "NORMAL (below threshold)"}')
    if has_det:
        ox1, oy1, ox2, oy2 = Preprocessor.unmap_box(rep_det['bbox'], rep_meta['s'], rep_meta['pad_x'], rep_meta['pad_y'], test_H, test_W)
        L.append(f'Bounding Box (x,y,w,h) [original px]: ({ox1:.0f}, {oy1:.0f}, {ox2 - ox1:.0f}, {oy2 - oy1:.0f})')
        L.append(f'Box Size (px)      : {ox2 - ox1:.0f} x {oy2 - oy1:.0f} ({100 * (ox2 - ox1) / test_W:.0f}% of width)')
    else:
        L.append('Bounding Box       : n/a (no detection)'); L.append('Box Size (px)      : n/a')
    L.append(f'Pyramid Levels Built: {rep_res["n_levels"]}')
    L.append(f'Raw Detections     : {rep_res["n_raw"]}')
    L.append(f'NMS Suppressed     : {rep_res["n_raw"] - len(rep_res["final"])}')
    L.append(f'Final Detections   : {len(rep_res["final"])}')

    # Block 2 - score decomposition
    L += ['', 'BLOCK 2 - SCORE DECOMPOSITION', SEP]
    if has_det:
        d = rep_det['decomp']; comps = [('Root Filter Score', d['root'])] + [(f'Part {i} Score', v) for i, v in enumerate(d['parts'])]
        tot_abs = sum(abs(v) for _, v in comps) or 1.0
        for name, v in comps:
            L.append(f'{name:<18}: {v:+9.4f}  ({100 * abs(v) / tot_abs:5.1f}%)  {bar(abs(v) / tot_abs)}')
        L += [SEP, f'Total Part Score        : {sum(d["parts"]):+.4f}', f'Total Deformation Cost  : {-sum(d["deform_cost"]):+.4f}  (penalty)',
              f'Bias Term               : {d["bias"]:+.4f}', SEP, f'FINAL DPM SCORE         : {d["total"]:+.4f}']
    else:
        L.append('n/a (no detection)')

    # Block 3 - part-level details
    L += ['', 'BLOCK 3 - PART-LEVEL DETAILS', SEP]
    if has_det:
        for i in range(N_PARTS):
            pb = [int(round(v)) for v in rep_det['part_boxes'][i]]; dx, dy = rep_det['deltas'][i]; c = rep_det['decomp']['deform_cost'][i]
            L.append(f'Part {i} | {PART_NAMES[i]:<34} | BBox(x,y,x2,y2)={tuple(pb)} | disp(dx,dy)=({dx:+d},{dy:+d}) cells | '
                     f'deform cost={c:+.4f}{"  HIGH" if abs(c) > HIGH_DEFORM_THRESH else ""}')
    else:
        L.append('n/a')

    # Block 4 - visualisation index
    L += ['', 'BLOCK 4 - VISUALISATION OUTPUT INDEX', SEP]
    index = [('class_distribution.png', 'Images per class'), ('box_scaling_check.png', 'Box-scaling verification on the PNGs'),
             ('sample_images.png', 'Sample positives / negatives'), ('preprocessing_comparison.png', 'Squashed vs aspect-restored vs CLAHE'),
             ('hog_visualization.png', 'HOG feature maps'), ('image_pyramid.png', 'Image pyramid levels'),
             ('root_filter_weights.png', 'Learned root filter'), ('evaluation_curves.png', 'ROC / PR / confusion / score histogram'),
             ('failure_cases.png', 'False positives and false negatives'), ('interpretability_4panel.png', 'Top true positive, 4-panel'),
             ('score_decomposition.png', 'Score decomposition (val example)'), ('part_activations.png', 'Part activations (val example)'),
             ('baseline_comparison.png', 'Baselines vs DPM')]
    index += [(p.name, 'THIS REPORT: ' + k) for k, p in FIG_PATHS.items()]
    for fn, desc in index:
        if (PLOTS_DIR / fn).exists():
            L.append(f'{str(PLOTS_DIR / fn):<75} {desc}')

    # Block 5 - computational profile
    L += ['', 'BLOCK 5 - COMPUTATIONAL PROFILE', SEP]
    for k in ('preprocess', 'pyramid', 'hog', 'root_scoring', 'nms', 'part_scoring', 'response_map', 'figures', 'total_pipeline'):
        L.append(f'{k:<16}: {T.get(k, 0.0):9.2f} ms')
    L.append(f'Peak host RAM    : {REPORT_PEAK_RSS_MB:.0f} MB')
    L.append(f'Peak GPU memory  : {"n/a (CPU)" if DEVICE == "cpu" else f"{REPORT_PEAK_GPU_MB:.0f} MB"}')
    L.append(f'Model size       : {BENCH_STATS["model_kb"]:.1f} KB')
    L.append(f'Est. FLOPs/image : {BENCH_STATS["flops"] / 1e6:.1f} M (root + part scoring)')

    # Block 6 - interpretability summary
    L += ['', 'BLOCK 6 - INTERPRETABILITY SUMMARY', SEP]
    if has_det:
        d = rep_det['decomp']; parts = np.array(d['parts']); costs = np.array(d['deform_cost'])
        best, least, mov = int(parts.argmax()), int(np.abs(parts).argmin()), int(np.abs(costs).argmax())
        n_arrows = sum(1 for dd in rep_det['deltas'] if dd != (0, 0))
        L += [f'Most Discriminative Part : Part {best} - {PART_NAMES[best]} ({parts[best]:+.3f})',
              f'Least Active Part        : Part {least} - {PART_NAMES[least]} ({parts[least]:+.3f})',
              f'Highest Deformation Part : Part {mov} - {PART_NAMES[mov]} (cost {costs[mov]:+.4f})',
              f'Deformation Arrows Drawn : {n_arrows}', f'Root Heatmap Generated   : {"Yes" if rep_map is not None else "No"}',
              'Score Breakdown Available: Yes', '']
        verdict = (f'The image was FLAGGED: its final DPM score ({d["total"]:+.2f}) is above the decision threshold ({THRESHOLD:+.2f}).'
                   if flagged else
                   f'The image was NOT flagged: its final DPM score ({d["total"]:+.2f}) does not exceed the decision threshold ({THRESHOLD:+.2f}).')
        L.append(verdict)
        L.append(f'The root template (global heart shape) contributed {d["root"]:+.2f}, and the four part filters together contributed {parts.sum():+.2f}; '
                 f'the strongest part was "{PART_NAMES[best]}" ({parts[best]:+.2f}) while "{PART_NAMES[least]}" was least active ({parts[least]:+.2f}).')
        L.append(f'The parts paid a total deformation cost of {costs.sum():.3f}, highest for "{PART_NAMES[mov]}", '
                 f'which indicates {"a border that sits far from its expected position" if abs(costs[mov]) > HIGH_DEFORM_THRESH else "that all parts stayed close to their expected positions"}.')
        L.append('Terms sum with the bias to the final score, so each number above is exactly how much that component pushed the decision.')
    else:
        L.append('No candidate window passed the position prior / pyramid constraints, so there is nothing to explain; the image is treated as normal.')

    # Block 7 - all scored detections
    L += ['', 'BLOCK 7 - ALL SCORED DETECTIONS', SEP, f'{"Det#":<5}{"BBox(x,y,w,h) cache px":<30}{"Root":>9}{"DPM":>9}  Classification']
    for j, dd in enumerate(sorted(rep_res['final'], key=lambda z: -z['dpm_score'])):
        x1, y1, x2, y2 = dd['bbox']
        L.append(f'{j:<5}({x1:.0f}, {y1:.0f}, {x2 - x1:.0f}, {y2 - y1:.0f})'.ljust(35) +
                 f'{dd["root_score"]:>9.3f}{dd["dpm_score"]:>9.3f}  {"ANOMALY" if dd["dpm_score"] > THRESHOLD else "normal"}')
    if not rep_res['final']:
        L.append('(no detections)')

    # Block 8 - clinical flag
    L += ['', 'BLOCK 8 - CLINICAL FLAG', LINE]
    L.append('WARNING  CLINICAL FLAG - ANOMALY DETECTED  WARNING' if flagged else 'OK  NORMAL')
    L += [LINE, f'Finding            : {TARGET_CLASS if flagged else "No cardiomegaly pattern detected"}',
          f'Confidence Level   : {conf}', f'Priority           : {priority}',
          'Recommended Action : ' + ('Prompt radiologist review of the highlighted cardiac region.' if priority == 'URGENT' else
                                     'Include in the routine radiologist worklist for review.' if flagged else
                                     'No action from this tool; follow the standard reading workflow.'),
          '', 'DISCLAIMER: This output is a research pre-filtering tool only.', 'All flagged cases must be reviewed by a qualified radiologist.',
          'This output does not constitute a clinical diagnosis.', '']
    L += [LINE, f'Model: {MODEL_VERSION} | Hardware: {hw} | RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB | '
                f'Time: {T["total_pipeline"]:.0f} ms | Scales: {rep_res["n_levels"]} | Detections: {len(rep_res["final"])}']
    return L

REPORT_PATH = REPORTS_DIR / f'report_{TARGET_CLASS}_{REPORT_ID}.txt'
_lines = build_report()
_lines.append(f'Saved path: {REPORT_PATH}'); _lines.append(LINE)
REPORT_TEXT = '\n'.join(_lines)
REPORT_PATH.write_text(REPORT_TEXT, encoding='utf-8')
print(REPORT_TEXT)

## Expected Behaviour & Honest Caveats

- **Cardiomegaly is a global measurement** (heart width relative to thorax width). A local HOG template (even with context margin and parts) can capture it only partly,
  so expect **modest performance compared with deep models**. The Section 12 comparison is the honest test of whether the parts add value over the root filter and over a plain image-level classifier. Report whatever it shows.
- **Resolution.** Cached images are 512 px and the root filter runs at 256 px. If part detail looks too coarse, raise `PART_RES_FACTOR`; if memory is tight, reduce it (and `LSVM_NEG_IMAGES`).
- **Aspect restoration is only as exact as `train_meta.csv`.** Section 2 verifies box bounds and draws the boxes; if that check passes the geometry is trustworthy.
- **Label noise remains.** Positives need two radiologists, but their boxes still differ; the consensus box is an average, so IoU of 0.5 is a demanding localisation target.
- **Scale coverage.** The root window is sized to the small end of heart sizes and the pyramid only shrinks the image. Section 5 prints the coverage; hearts smaller than the window cannot be matched.
- **Threshold.** `THRESHOLD = 0` is not tuned. The Youden threshold in Section 10 is tuned on validation data and is optimistic; for a real screening use, tune on a separate calibration set.
- **Not a medical device.** The report is a research pre-filter and must never replace review by a qualified radiologist.